# Dictionary Discovery Workflow v4 - Complete Implementation

## Overview
Systematic dictionary-based topic discovery and model training workflow with structured file organization.

### Checkpoints:
- **CHECKPOINT 0**: Initial Setup → Create folders, load config
- **CHECKPOINT 1**: Text Processing → Chunk corpus
- **CHECKPOINT 2**: Vocabulary Building → Build vocab from chunks
- **CHECKPOINT 3**: Dictionary Expansion → Expand keywords (⚠️ MANUAL CURATION REQUIRED)
- **CHECKPOINT 4**: Topic Vectors → Build weighted topic vectors
- **CHECKPOINT 5**: Chunk Scoring → Score & classify by confidence
- **CHECKPOINT 6**: Training Data Prep → Create train/val splits
- **CHECKPOINT 7**: Model Training → Train BERTJE
- **CHECKPOINT 8**: BERTJE Labeling → Label corpus with trained model
- **CHECKPOINT 9**: Visualizations → Generate clustering & performance plots

### Folder Structure:
```
workflow_data/{ModelType}-{Topic}_{Date}_{Version}/
  ├── config/
  ├── Dictionary/
  │   └── Dictionary_suggestions/
  ├── Model_finetuning/
  ├── Cosine_labeling/
  ├── Bertje_labeling/
  ├── Visuals/
  └── Other_data/
```

---
# CHECKPOINT 0: Initial Setup
---

In [116]:
# ============================================================
# CELL 0.1: IMPORTS
# ============================================================
import os
import re
import json
import hashlib
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

# NLTK
import nltk
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# ML libraries
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer

# Suppress warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

print("✓ All imports successful")

✓ All imports successful


In [120]:
# ============================================================
# CONFIGURATION WITH CORPUS FILTERING
# ============================================================

CONFIG = {
    # =====================
    # WORKFLOW METADATA
    # =====================
    "workflow": {
        # Model type: "Pretrained" or "Finetuned_{source}"
        "model_type": "Pretrained_Slavery",
        
        # Topic: "Slavery", "Policy", "Slavery-Policy", etc.
        "topic": "Slavery",
        
        # Version (None for auto-increment)
        "version": None,
    },

    # =====================
    # PATHS
    # =====================
    "paths": {
        "corpus_dir": "Slavery_text",
        "dictionary_excel": "dutch_slavery_legacy_dictionary.xlsx",
        "workflow_base": "workflow_data",
        "pretrained_model_path": "workflow_data\\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v1\\Model_finetuning",
    },
    
    # =====================
    # CORPUS FILTERING (NEW SECTION)
    # =====================
    "corpus_filter": {
        # Enable/disable filtering
        "enabled": False,
        
        # Year filtering options:
        # Option 1: Specific years as a list
        "years": [2022],
        
        # Option 2: Year range (overrides "years" if set)
        # Set to None to disable, or use tuple like (2015, 2023)
        "year_range": None,  # e.g., (2015, 2023) for 2015 through 2023
        
        # Option 3: Only documents after/before certain year
        "year_min": None,  # e.g., 2000 for documents from 2000 onwards
        "year_max": None,  # e.g., 2023 for documents up to 2023
        
        # Document type filtering
        # List of document types to include (None = all types)
        "doc_types": ["beleidsnotas", "jaarplannen","besluiten","jaarverslagen"],  # e.g., ["policy", "report", "legislation"]
        
        # Filename pattern matching (uses regex)
        # List of patterns - documents matching ANY pattern are included
        "filename_patterns": None,  # e.g., [".*gemeente.*", ".*ministry.*"]
        
        # Exclude patterns (uses regex)
        # Documents matching ANY exclude pattern are skipped
        "exclude_patterns": None,  # e.g., [".*draft.*", ".*concept.*"]
    },
    
    # =====================
    # MODEL SETTINGS
    # =====================
    "model": {
        "base_model_name": "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers",
        "use_pretrained": False,
    },
    
    # =====================
    # DICTIONARY SETTINGS
    # =====================
    "dictionary": {
        "use_excel": True,
        "topic_column": "topic",
        "keyword_column": "keyword",
        "sheet_name": 0,
        "default_topics": {
            "Historical slavery": ["slavernij", "tot-slaaf-gemaakte", "dwangarbeid", "zweep"],
            "Colonialism": ["kolonie", "koloniaal", "voc", "wic", "exploitatie"],
            "Modern racism& inequality": ["racisme", "discriminatie", "ongelijkheid"],
        },
    },
    
    # =====================
    # TEXT PROCESSING
    # =====================
    "chunking": {
        "sentences_per_chunk": 10,
        "min_sentences_to_keep": 3,
        "drop_likely_english": True,
        "remove_stopwords": True,
        "use_stemming": False,
    },
    
    "tokenize": {
        "lower": True,
        "keep_hyphen": True,
        "min_len": 2,
        "max_len": 30,
        "pattern": r"[0-9A-Za-zÀ-ÖØ-öø-ÿ\-]+",
    },
    
    # =====================
    # VOCABULARY SETTINGS
    # =====================
    "vocab": {
        "min_df": 5,
        "max_vocab": 50000,
    },
    
    # =====================
    # EXPANSION SETTINGS
    # =====================
    "expand": {
        "k_nearest": 50,
        "topN_per_topic": 300,
        "min_cosine": 0.55,
    },
    
    # =====================
    # SCORING SETTINGS
    # =====================
    "scoring": {
        "use_sif": True,
        "sif_a": 1e-3,
        "high_confidence_score": 0.50,
        "high_confidence_margin": 0.05,
        "low_confidence_score": 0.40,
        "low_confidence_margin": 0.02,
    },
    
    # =====================
    # TRAINING SETTINGS
    # =====================
    "training": {
        "num_epochs": 5,
        "batch_size_train": 16,
        "batch_size_eval": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "dataset_option": "option2",
    },
}

print("✓ Configuration loaded")
print(f"\nWorkflow: {CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}")

# Display active filters
if CONFIG['corpus_filter']['enabled']:
    print("\n📂 Corpus Filtering ENABLED:")
    
    # Year filters
    if CONFIG['corpus_filter']['year_range']:
        print(f"  - Year range: {CONFIG['corpus_filter']['year_range'][0]} to {CONFIG['corpus_filter']['year_range'][1]}")
    elif CONFIG['corpus_filter']['years']:
        print(f"  - Specific years: {CONFIG['corpus_filter']['years']}")
    elif CONFIG['corpus_filter']['year_min'] or CONFIG['corpus_filter']['year_max']:
        if CONFIG['corpus_filter']['year_min']:
            print(f"  - Year minimum: {CONFIG['corpus_filter']['year_min']}")
        if CONFIG['corpus_filter']['year_max']:
            print(f"  - Year maximum: {CONFIG['corpus_filter']['year_max']}")
    
    # Document type filters
    if CONFIG['corpus_filter']['doc_types']:
        print(f"  - Document types: {CONFIG['corpus_filter']['doc_types']}")
    
    # Pattern filters
    if CONFIG['corpus_filter']['filename_patterns']:
        print(f"  - Include patterns: {CONFIG['corpus_filter']['filename_patterns']}")
    if CONFIG['corpus_filter']['exclude_patterns']:
        print(f"  - Exclude patterns: {CONFIG['corpus_filter']['exclude_patterns']}")
else:
    print("\n📂 Corpus Filtering DISABLED - loading all documents")

✓ Configuration loaded

Workflow: Pretrained_Slavery-Slavery

📂 Corpus Filtering DISABLED - loading all documents


In [91]:
# ============================================================
# CELL 0.3: FILE SYSTEM UTILITIES
# ============================================================

class WorkflowFileSystem:
    """Manages structured folder system for workflow data."""
    
    def __init__(self, config):
        self.config = config
        self.root = None
        self.folders = {}
    
    def create_workflow_folder(self):
        """Create main workflow folder with subfolders."""
        model_type = self.config["workflow"]["model_type"]
        topic = self.config["workflow"]["topic"]
        date = datetime.now().strftime("%m.%d.%y")
        
        version = self.config["workflow"]["version"]
        if version is None:
            version = self._get_next_version(model_type, topic, date)
        
        folder_name = f"{model_type}-{topic}_{date}_{version}"
        base_dir = self.config["paths"]["workflow_base"]
        self.root = Path(base_dir) / folder_name
        self.root.mkdir(parents=True, exist_ok=True)
        
        subfolder_names = [
            "config",
            "Dictionary",
            "Dictionary/Dictionary_suggestions",
            "Model_finetuning",
            "Cosine_labeling",
            "Bertje_labeling",
            "Visuals",
            "Other_data",
        ]
        
        for subfolder in subfolder_names:
            path = self.root / subfolder
            path.mkdir(parents=True, exist_ok=True)
            key = subfolder.split("/")[-1]
            self.folders[key] = path
        
        self.folders["Dictionary"] = self.root / "Dictionary"
        
        print(f"\n{'='*60}")
        print("WORKFLOW FOLDER CREATED")
        print(f"{'='*60}")
        print(f"Location: {self.root}")
        print(f"\nSubfolders:")
        for name in subfolder_names:
            print(f"  ✓ {name}/")
        
        return self.root
    
    def _get_next_version(self, model_type, topic, date):
        """Auto-increment version number."""
        base_dir = Path(self.config["paths"]["workflow_base"])
        if not base_dir.exists():
            return "v1"
        
        prefix = f"{model_type}-{topic}_{date}_v"
        existing = [d.name for d in base_dir.iterdir() if d.is_dir() and d.name.startswith(prefix)]
        
        if not existing:
            return "v1"
        
        versions = []
        for folder in existing:
            try:
                version_str = folder.split("_v")[-1]
                versions.append(int(version_str.replace("v", "")))
            except:
                continue
        
        if versions:
            return f"v{max(versions) + 1}"
        return "v1"
    
    def load_existing_workflow(self, folder_path):
        """Load existing workflow folder."""
        self.root = Path(folder_path)
        if not self.root.exists():
            raise ValueError(f"Workflow folder not found: {folder_path}")
        
        subfolder_names = [
            "config", "Dictionary", "Dictionary_suggestions",
            "Model_finetuning", "Cosine_labeling", "Bertje_labeling",
            "Visuals", "Other_data"
        ]
        
        for name in subfolder_names:
            if name == "Dictionary_suggestions":
                path = self.root / "Dictionary" / name
            else:
                path = self.root / name
            if path.exists():
                self.folders[name] = path
        
        print(f"✓ Loaded existing workflow: {self.root.name}")
        return self.root
    
    def save_config(self, checkpoint_name=None):
        """Save CONFIG to config folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"config_{checkpoint_name}_{timestamp}.json" if checkpoint_name else f"config_{timestamp}.json"
        config_path = self.folders["config"] / filename
        
        config_data = {
            "metadata": {
                "timestamp": timestamp,
                "checkpoint": checkpoint_name,
                "workflow_folder": str(self.root),
            },
            "config": self.config
        }
        
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Config saved: {config_path.name}")
        return config_path
    
    def save_data(self, data, filename, folder_key, file_format="csv"):
        """Save data to specific folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        full_filename = f"{filename}.{file_format}"
        filepath = folder / full_filename
        
        if file_format == "csv":
            if not isinstance(data, pd.DataFrame):
                raise ValueError("CSV format requires DataFrame")
            data.to_csv(filepath, index=False, encoding='utf-8')
        elif file_format == "json":
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        elif file_format == "npy":
            np.save(filepath, data, allow_pickle=True)
        else:
            raise ValueError(f"Unsupported format: {file_format}")
        
        print(f"✓ Saved: {folder_key}/{full_filename}")
        return filepath
    
    def copy_file_to_folder(self, source_path, folder_key, new_name=None):
        """Copy external file to workflow folder."""
        if self.root is None:
            raise ValueError("Workflow folder not initialized")
        
        folder = self.folders.get(folder_key)
        if folder is None:
            raise ValueError(f"Unknown folder key: {folder_key}")
        
        source = Path(source_path)
        if not source.exists():
            raise FileNotFoundError(f"Source file not found: {source_path}")
        
        dest_name = new_name if new_name else source.name
        dest_path = folder / dest_name
        
        shutil.copy2(source, dest_path)
        print(f"✓ Copied: {source.name} → {folder_key}/{dest_name}")
        return dest_path

print("✓ WorkflowFileSystem class defined")

✓ WorkflowFileSystem class defined


In [92]:
# ============================================================
# CELL 0.4: CREATE OR LOAD WORKFLOW
# ============================================================

# Choose one:
CREATE_NEW = True  # Set to False to load existing
EXISTING_FOLDER = r"C:\Users\Home\policy-analysis\workflow_data\Finetuned_Slavery-Slavery-policy_10.28.25_v1"  # Set path if loading existing

fs = WorkflowFileSystem(CONFIG)

if CREATE_NEW:
    workflow_root = fs.create_workflow_folder()
    fs.save_config("initial_setup")
else:
    if EXISTING_FOLDER is None:
        raise ValueError("EXISTING_FOLDER must be set when CREATE_NEW=False")
    workflow_root = fs.load_existing_workflow(EXISTING_FOLDER)

print(f"\n✓ Workflow initialized: {workflow_root}")


WORKFLOW FOLDER CREATED
Location: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1

Subfolders:
  ✓ config/
  ✓ Dictionary/
  ✓ Dictionary/Dictionary_suggestions/
  ✓ Model_finetuning/
  ✓ Cosine_labeling/
  ✓ Bertje_labeling/
  ✓ Visuals/
  ✓ Other_data/
✓ Config saved: config_initial_setup_20251030_170848.json

✓ Workflow initialized: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1


In [93]:
# ============================================================
# CELL 0.5: LOAD DICTIONARY
# ============================================================

def load_dictionary_from_excel(excel_path, config):
    """Load topics and keywords from Excel."""
    if not Path(excel_path).exists():
        print(f"⚠ Excel not found: {excel_path}")
        return config["dictionary"]["default_topics"]
    
    try:
        df = pd.read_excel(excel_path, sheet_name=config["dictionary"]["sheet_name"])
        topic_col = config["dictionary"]["topic_column"]
        keyword_col = config["dictionary"]["keyword_column"]
        
        if topic_col not in df.columns or keyword_col not in df.columns:
            return config["dictionary"]["default_topics"]
        
        topics_dict = {}
        for topic, group in df.groupby(topic_col):
            keywords = group[keyword_col].dropna().str.strip().tolist()
            if keywords:
                topics_dict[topic] = keywords
        
        print(f"✓ Loaded from Excel: {len(topics_dict)} topics, {len(df)} keywords")
        return topics_dict
    except Exception as e:
        print(f"⚠ Error: {e}")
        return config["dictionary"]["default_topics"]

if CONFIG["dictionary"]["use_excel"]:
    topics = load_dictionary_from_excel(CONFIG["paths"]["dictionary_excel"], CONFIG)
    CONFIG["topics"] = topics
    if Path(CONFIG["paths"]["dictionary_excel"]).exists():
        fs.copy_file_to_folder(
            CONFIG["paths"]["dictionary_excel"],
            "Dictionary",
            "input_dictionary.xlsx"
        )
else:
    CONFIG["topics"] = CONFIG["dictionary"]["default_topics"]

print(f"\n{'='*60}")
print("TOPICS LOADED")
print(f"{'='*60}")
for topic, keywords in CONFIG["topics"].items():
    print(f"  {topic}: {len(keywords)} keywords")

fs.save_config("with_dictionary")

✓ Loaded from Excel: 3 topics, 55 keywords
✓ Copied: dutch_slavery_legacy_dictionary.xlsx → Dictionary/input_dictionary.xlsx

TOPICS LOADED
  Colonialism: 19 keywords
  Historical Slavery: 18 keywords
  Modern Racism & Inequality: 18 keywords
✓ Config saved: config_with_dictionary_20251030_170853.json


WindowsPath('workflow_data/Pretrained_Slavery-Slavery_10.30.25_v1/config/config_with_dictionary_20251030_170853.json')

In [94]:
# ============================================================
# CELL 3.1: LOAD SENTENCE TRANSFORMER MODEL
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 3 START - LOADING VOCABULARY & MODEL")
print(f"{'='*60}")


print(f"\n{'='*60}")
print("LOADING SENTENCE TRANSFORMER MODEL")
print(f"{'='*60}")

if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
    model_path = CONFIG['paths']['pretrained_model_path']
    if Path(model_path).exists():
        st_model = SentenceTransformer(model_path)
        print(f"✓ Loaded pretrained model from: {model_path}")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"⚠ Pretrained path not found, using base model")
else:
    st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
    print(f"✓ Loaded base model: {CONFIG['model']['base_model_name']}")

def st_embed(texts: list, batch_size: int = 256) -> np.ndarray:
    return st_model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=False, 
        normalize_embeddings=True
    )

print(f"\n✓ Model ready")
print(f"  Max sequence length: {st_model.max_seq_length}")
print(f"  Embedding dimension: {st_model.get_sentence_embedding_dimension()}")



CHECKPOINT 3 START - LOADING VOCABULARY & MODEL

LOADING SENTENCE TRANSFORMER MODEL
✓ Loaded base model: NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers

✓ Model ready
  Max sequence length: 128
  Embedding dimension: 768


✅ **CHECKPOINT 0 COMPLETE** - Folder structure created, dictionary loaded

---
# CHECKPOINT 1: Text Processing
---

Chunks corpus into sentence-based segments with cleaning.

In [95]:
# ============================================================
# CELL 1.1: TEXT CLEANING UTILITIES
# ============================================================

stemmer = SnowballStemmer("dutch")

nltk_stopwords = set(stopwords.words('dutch')) | set(stopwords.words('english'))
custom_stopwords = set([
    "de","het","een","en","van","in","op","met","voor","tegen","zonder","bij",
    "naar","tot","uit","door","aan","om","te","als","ook","maar","want","dus",
    "of","dan","nog","wel","zijn","is","was","waren","worden","hebben","heeft",
    "had","doet","doen","al","alle","meer","minder","veel","weinig","binnen",
    "buiten","tussen","onder","boven","over","na","achter","naast","sinds",
    "tijdens","zoals","ik","jij","hij","zij","wij","jullie","u","je","ze",
    "dit","dat","die","deze","welke","ons","hun","hem","haar","bijlage",
    "bijlagen","inleiding","samenvatting","conclusie","conclusies","jaar","jaren",
])
ALL_STOPWORDS = nltk_stopwords | custom_stopwords

ENGLISH_HINTS = set("the and of to in that is for on with as by from at it this be are were was has have will would can could should".split())
DUTCH_HINTS = set("de het een en van voor met op aan te is zijn worden was waren niet bij in over uit door naar tot als ook om".split())

def likely_english_sentence(s: str) -> bool:
    if not CONFIG["chunking"]["drop_likely_english"]:
        return False
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", s.lower())
    if not tokens:
        return False
    e = sum(t in ENGLISH_HINTS for t in tokens)
    d = sum(t in DUTCH_HINTS for t in tokens)
    return e > max(2, d + 1)

def remove_stopwords_and_numbers(text: str) -> str:
    if pd.isna(text):
        return ""
    tokens = re.findall(r"\b\w+\b", text.lower())
    filtered = [tok for tok in tokens if tok not in ALL_STOPWORDS and not tok.isdigit()]
    return " ".join(filtered)

def stem_text(text: str) -> str:
    tokens = re.findall(r"\b\w+\b", text.lower())
    return " ".join(stemmer.stem(w) for w in tokens)

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def short_file_hash(path: str, n=8) -> str:
    return hashlib.sha1(path.encode("utf-8", errors="ignore")).hexdigest()[:n]

def make_chunk_uid(file_path: str, chunk_idx: int) -> str:
    return f"{short_file_hash(file_path)}:{chunk_idx:05d}"

print("✓ Text cleaning utilities ready")

✓ Text cleaning utilities ready


In [96]:
# ============================================================
# METADATA EXTRACTION FROM NESTED FOLDER STRUCTURE
# ============================================================

def extract_path_metadata(file_path: str, corpus_dir: Path) -> dict:
    """
    Extract metadata from nested folder structure.
    
    Expected structure: corpus_dir/doc_type/year/document_folder/file.txt
    Example: Policyarchive_text/convenanten/2019/document-name/document-name.2.txt
    
    Args:
        file_path: Full path to the document file
        corpus_dir: Base directory of the corpus
    
    Returns:
        Dictionary with doc_type, year, document_folder, and filename
        - doc_type: The type of document (e.g., 'convenanten', 'kamerstukken')
        - year: The year extracted from the path (if parseable)
        - document_folder: The parent folder containing the document
        - filename: Just the filename
        - relative_path: Path relative to corpus_dir
    """
    try:
        # Get the path relative to the corpus directory
        # This removes the corpus_dir prefix so we can analyze the structure
        rel_path = Path(file_path).relative_to(corpus_dir)
        parts = rel_path.parts  # Split path into components
        
        # Initialize metadata dictionary with None values
        metadata = {
            'doc_type': None,           # Changed from 'category'
            'year': None,
            'document_folder': None,
            'filename': rel_path.name,
            'relative_path': str(rel_path)
        }
        
        # Extract doc_type from first level of folder structure
        # Example: if path is "convenanten/2019/doc/file.txt", doc_type = "convenanten"
        if len(parts) >= 2:
            metadata['doc_type'] = parts[0]  # Changed from 'category'
        
        # Extract year from second level (if it's a 4-digit number)
        if len(parts) >= 3:
            try:
                year_candidate = parts[1]
                # Check if it's a valid 4-digit year
                if year_candidate.isdigit() and len(year_candidate) == 4:
                    metadata['year'] = int(year_candidate)
            except:
                pass  # If parsing fails, year stays None
        
        # Extract document folder (the parent folder of the file)
        if len(parts) >= 4:
            metadata['document_folder'] = parts[-2]  # Second to last is document folder
        elif len(parts) == 3:
            metadata['document_folder'] = parts[-2]  # If only 3 levels, still use parent folder
        
        return metadata
        
    except Exception as e:
        # Fallback if path parsing fails - return basic info only
        return {
            'doc_type': None,
            'year': None,
            'document_folder': None,
            'filename': Path(file_path).name,
            'relative_path': str(Path(file_path).name)
        }

In [97]:
# ============================================================
# DOCUMENT FILTERING LOGIC
# ============================================================

def should_include_document(file_path: str, corpus_dir: Path, filter_config: dict) -> tuple:
    """
    Determine if a document should be included based on filter settings.
    
    This function checks the nested folder structure and filename against
    all configured filters to decide if a document should be processed.
    
    Args:
        file_path: Full path to the document file
        corpus_dir: Base corpus directory path
        filter_config: The corpus_filter section from CONFIG
        
    Returns:
        Tuple of (should_include: bool, reason: str, metadata: dict)
        - should_include: True if document passes all filters
        - reason: Explanation of why document was included/excluded
        - metadata: Document metadata extracted from path
    """
    # If filtering is disabled in config, include everything
    if not filter_config.get('enabled', False):
        metadata = extract_path_metadata(file_path, corpus_dir)
        return True, "filtering disabled", metadata
    
    # Extract metadata from the file path
    metadata = extract_path_metadata(file_path, corpus_dir)
    
    # Get relative path for pattern matching
    rel_path = metadata['relative_path']
    
    # ===== STEP 1: CHECK EXCLUDE PATTERNS FIRST =====
    # These take highest priority - if path matches any exclude pattern, skip it
    exclude_patterns = filter_config.get('exclude_patterns')
    if exclude_patterns:
        for pattern in exclude_patterns:
            if re.search(pattern, rel_path, re.IGNORECASE):
                return False, f"matches exclude pattern: {pattern}", metadata
    
    # ===== STEP 2: CHECK DOC_TYPE FILTERS =====
    # Changed from category to doc_type
    
    # Check if doc_type is in exclude list
    exclude_doc_types = filter_config.get('exclude_doc_types')  # Changed from 'exclude_categories'
    if exclude_doc_types and metadata['doc_type'] in exclude_doc_types:
        return False, f"doc_type '{metadata['doc_type']}' is excluded", metadata
    
    # Check if doc_type is in include list (if specified)
    doc_types = filter_config.get('doc_types')  # Changed from 'categories'
    if doc_types:
        if metadata['doc_type'] is None:
            # If doc_type is required but not found, exclude document
            if filter_config.get('require_doc_type', False):  # Changed from 'require_category'
                return False, "no doc_type found in path", metadata
        elif metadata['doc_type'] not in doc_types:
            return False, f"doc_type '{metadata['doc_type']}' not in allowed doc_types", metadata
    
    # ===== STEP 3: CHECK YEAR FILTERS =====
    year = metadata['year']
    
    # Check if year is required but missing
    if filter_config.get('require_year', False) and year is None:
        return False, "year required but not found in path", metadata
    
    # If year exists, check against year filters
    if year is not None:
        # Check year range (highest priority if specified)
        year_range = filter_config.get('year_range')
        if year_range:
            if not (year_range[0] <= year <= year_range[1]):
                return False, f"year {year} outside range {year_range}", metadata
        
        # Check specific years list
        elif filter_config.get('years'):
            if year not in filter_config['years']:
                return False, f"year {year} not in allowed years", metadata
        
        # Check min/max years
        else:
            year_min = filter_config.get('year_min')
            year_max = filter_config.get('year_max')
            
            if year_min and year < year_min:
                return False, f"year {year} before minimum {year_min}", metadata
            if year_max and year > year_max:
                return False, f"year {year} after maximum {year_max}", metadata
    
    # ===== STEP 4: CHECK DOCUMENT FOLDER FILTERS =====
    document_folders = filter_config.get('document_folders')
    if document_folders:
        if metadata['document_folder'] is None:
            return False, "no document folder found in path", metadata
        if metadata['document_folder'] not in document_folders:
            return False, f"document folder '{metadata['document_folder']}' not in allowed folders", metadata
    
    # ===== STEP 5: CHECK FILENAME PATTERNS =====
    filename_patterns = filter_config.get('filename_patterns')
    if filename_patterns:
        matched = False
        for pattern in filename_patterns:
            if re.search(pattern, metadata['filename'], re.IGNORECASE):
                matched = True
                break
        if not matched:
            return False, "filename doesn't match any include pattern", metadata
    
    # If we've made it through all filters, include the document
    return True, "passed all filters", metadata

In [98]:
# ============================================================
# MAIN CORPUS PROCESSING WITH FILTERING
# ============================================================

print(f"\n{'='*60}")
print("PROCESSING CORPUS (WITH FILTERING)")
print(f"{'='*60}")

# Initialize list to store all chunks from all documents
all_chunks = []
corpus_dir = Path(CONFIG["paths"]["corpus_dir"])

# Check if corpus directory exists
if not corpus_dir.exists():
    print(f"⚠ Corpus directory not found: {corpus_dir}")
else:
    # Recursively find all .txt files in nested folders
    # The ** pattern means "look in all subdirectories at any depth"
    all_doc_files = list(corpus_dir.glob("**/*.txt"))
    print(f"\nFound {len(all_doc_files)} total documents in nested folders")
    
    # Get filter configuration
    filter_config = CONFIG.get("corpus_filter", {"enabled": False})
    
    # ===== DISPLAY FILTER SETTINGS =====
    if filter_config.get('enabled', False):
        print("\n🔍 Applying filters...")
        
        # Display doc_type filters (changed from categories)
        print(f"  Doc types: {filter_config.get('doc_types', 'All')}")
        print(f"  Exclude doc types: {filter_config.get('exclude_doc_types', 'None')}")
        
        # Display year filters
        if filter_config.get('year_range'):
            print(f"  Year range: {filter_config['year_range'][0]} - {filter_config['year_range'][1]}")
        elif filter_config.get('years'):
            print(f"  Specific years: {filter_config['years']}")
        elif filter_config.get('year_min') or filter_config.get('year_max'):
            if filter_config.get('year_min'):
                print(f"  Year minimum: {filter_config['year_min']}")
            if filter_config.get('year_max'):
                print(f"  Year maximum: {filter_config['year_max']}")
        
        # Display other filters
        if filter_config.get('document_folders'):
            print(f"  Document folders: {filter_config['document_folders']}")
        if filter_config.get('filename_patterns'):
            print(f"  Filename patterns: {filter_config['filename_patterns']}")
        if filter_config.get('exclude_patterns'):
            print(f"  Exclude patterns: {filter_config['exclude_patterns']}")
    
    # ===== APPLY FILTERS TO ALL DOCUMENTS =====
    doc_files = []  # Documents that pass filters
    skipped_docs = []  # Documents that don't pass filters
    
    for doc_path in all_doc_files:
        include, reason, metadata = should_include_document(str(doc_path), corpus_dir, filter_config)
        if include:
            doc_files.append(doc_path)
        else:
            skipped_docs.append((doc_path.name, reason))
    
    # Display filtering results
    print(f"\n✅ {len(doc_files)} documents passed filters")
    if skipped_docs:
        print(f"⏭️  {len(skipped_docs)} documents skipped")
        # Show first 10 skipped documents as examples
        if len(skipped_docs) <= 10:
            print("\nSkipped documents:")
            for filename, reason in skipped_docs[:10]:
                print(f"  - {filename}: {reason}")
    
    # ===== ANALYZE FILTERED CORPUS STRUCTURE =====
    # Count documents by doc_type and year to show overview
    doc_types = {}  # Changed from categories
    years = {}
    
    for doc_path in doc_files:
        _, _, metadata = should_include_document(str(doc_path), corpus_dir, filter_config)
        
        # Count doc_types
        if metadata['doc_type']:
            doc_types[metadata['doc_type']] = doc_types.get(metadata['doc_type'], 0) + 1
        
        # Count years
        if metadata['year']:
            years[metadata['year']] = years.get(metadata['year'], 0) + 1
    
    # Display doc_type distribution (changed from categories)
    if doc_types:
        print(f"\n📁 Document types in filtered corpus:")
        for doc_type, count in sorted(doc_types.items()):
            print(f"  - {doc_type}: {count} documents")
    
    # Display year distribution
    if years:
        print(f"\n📅 Years in filtered corpus:")
        for year, count in sorted(years.items()):
            print(f"  - {year}: {count} documents")
    
    # ===== CHUNK ALL FILTERED DOCUMENTS =====
    print(f"\n{'='*60}")
    print("CHUNKING DOCUMENTS")
    print(f"{'='*60}\n")
    
    # Process each document that passed filters
    for doc_path in tqdm(doc_files, desc="Chunking documents"):
        # Read document text
        with open(doc_path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        
        # Extract metadata from path structure
        metadata = extract_path_metadata(str(doc_path), corpus_dir)
        
        # Split document into chunks
        doc_chunks = chunk_by_sentences(text, str(doc_path))
        
        # Store each chunk with its metadata
        for chunk_uid, raw_text, text_for_scoring, sentence_count in doc_chunks:
            all_chunks.append({
                'file_path': str(doc_path),
                'chunk_uid': chunk_uid,
                'raw_text': raw_text,
                'text_for_scoring': text_for_scoring,
                'sentence_count': sentence_count,
                # Metadata from folder structure (changed category to doc_type)
                'doc_type': metadata['doc_type'],  # Changed from 'category'
                'year': metadata['year'],
                'document_folder': metadata['document_folder'],
                'filename': metadata['filename']
            })
    
    # ===== CREATE DATAFRAME AND SAVE =====
    chunks_df = pd.DataFrame(all_chunks)
    fs.save_data(chunks_df, "chunked_corpus", "Other_data", "csv")
    
    # ===== DISPLAY FINAL STATISTICS =====
    print(f"\n{'='*60}")
    print("CHUNKING COMPLETE")
    print(f"{'='*60}")
    print(f"\n✓ Processed {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    print(f"  Avg sentences/chunk: {chunks_df['sentence_count'].mean():.1f}")
    print(f"  Empty scoring text: {(chunks_df['text_for_scoring'] == '').sum()}")
    
    # Show metadata distribution in chunks
    if 'doc_type' in chunks_df.columns:  # Changed from 'category'
        print(f"\n📊 Metadata distribution in chunks:")
        print(f"  Document types: {chunks_df['doc_type'].nunique()} unique")
        print(f"  Years: {chunks_df['year'].nunique()} unique")
        print(f"  Document folders: {chunks_df['document_folder'].nunique()} unique")
        
        # Show top doc_types by chunk count (changed from categories)
        if chunks_df['doc_type'].notna().any():
            print(f"\n  Top document types by chunk count:")
            for doc_type, count in chunks_df['doc_type'].value_counts().head(5).items():
                print(f"    - {doc_type}: {count} chunks")
        
        # Show year distribution in chunks
        if chunks_df['year'].notna().any():
            print(f"\n  Year distribution:")
            for year, count in chunks_df['year'].value_counts().sort_index().items():
                print(f"    - {year}: {count} chunks")
    
    # Save checkpoint
    fs.save_config("checkpoint1_chunks")


PROCESSING CORPUS (WITH FILTERING)

Found 1582 total documents in nested folders

✅ 1582 documents passed filters

CHUNKING DOCUMENTS



Chunking documents: 100%|██████████| 1582/1582 [00:01<00:00, 1180.50it/s]


✓ Saved: Other_data/chunked_corpus.csv

CHUNKING COMPLETE

✓ Processed 2900 chunks from 1378 documents
  Avg sentences/chunk: 8.5
  Empty scoring text: 59

📊 Metadata distribution in chunks:
  Document types: 0 unique
  Years: 0 unique
  Document folders: 0 unique
✓ Config saved: config_checkpoint1_chunks_20251030_170922.json


✅ **CHECKPOINT 1 COMPLETE** - Corpus chunked and saved

**Resume**: Load `chunks_df` from `Other_data/chunked_corpus.csv`

---
# CHECKPOINT 2: Vocabulary Building
---

Build vocabulary from corpus with frequency filtering.

In [100]:
# ============================================================
# CELL 2.1: TOKENIZATION FOR VOCAB BUILDING
# ============================================================

_tok_re = re.compile(CONFIG["tokenize"]["pattern"])

def tokenize(text: str) -> list:
    if CONFIG["tokenize"]["lower"]:
        text = text.lower()
    toks = _tok_re.findall(text)
    keep = []
    mn = CONFIG["tokenize"]["min_len"]
    mx = CONFIG["tokenize"]["max_len"]
    for t in toks:
        if not CONFIG["tokenize"]["keep_hyphen"]:
            t = t.replace("-", "")
        if mn <= len(t) <= mx:
            keep.append(t)
    return keep

def read_text(path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return path.read_text(errors="ignore")

print("✓ Tokenizer ready")

✓ Tokenizer ready


In [101]:
# ============================================================
# BUILD VOCABULARY FROM CHUNKED CORPUS (DIRECT PATH)
# ============================================================

print(f"\n{'='*60}")
print("BUILDING VOCABULARY FROM CHUNKED CORPUS")
print(f"{'='*60}")

# Try to construct the path directly from CONFIG
workflow_base = Path(CONFIG["paths"]["workflow_base"])
workflow_name = f"{CONFIG['workflow']['model_type']}-{CONFIG['workflow']['topic']}"

# Find the most recent workflow directory (or use a specific version)
if CONFIG['workflow']['version']:
    workflow_dir = workflow_base / f"{workflow_name}_{CONFIG['workflow']['version']}"
else:
    # Find the most recent version
    matching_dirs = list(workflow_base.glob(f"{workflow_name}_*"))
    if matching_dirs:
        workflow_dir = sorted(matching_dirs)[-1]  # Get the most recent
    else:
        print(f"❌ Error: No workflow directory found matching {workflow_name}")
        workflow_dir = None

if workflow_dir and workflow_dir.exists():
    chunked_corpus_path = workflow_dir / "Other_data" / "chunked_corpus.csv"
    print(f"📂 Using workflow directory: {workflow_dir}")
else:
    # Fallback: look in current working directory
    chunked_corpus_path = Path("Other_data") / "chunked_corpus.csv"
    print(f"📂 Using relative path: {chunked_corpus_path}")

if not chunked_corpus_path.exists():
    print(f"❌ Error: Chunked corpus not found at {chunked_corpus_path}")
    print("   Please run the chunking step first.")
    print(f"\n   Looking for files in: {chunked_corpus_path.parent}")
    if chunked_corpus_path.parent.exists():
        files = list(chunked_corpus_path.parent.glob("*.csv"))
        print(f"   Found {len(files)} CSV files:")
        for f in files[:10]:
            print(f"     - {f.name}")
else:
    # Load the chunks DataFrame
    print(f"✓ Loading from: {chunked_corpus_path}")
    chunks_df = pd.read_csv(chunked_corpus_path)
    
    print(f"\n✓ Loaded {len(chunks_df)} chunks from {chunks_df['file_path'].nunique()} documents")
    
    # Show filtering info if available
    if 'category' in chunks_df.columns and chunks_df['category'].notna().any():
        print(f"  Categories: {chunks_df['category'].nunique()} unique")
        print(f"  Years: {chunks_df['year'].nunique()} unique")
    
    # Build vocabulary from the processed text
    print("\n📝 Tokenizing chunks...")
    
    term_freq = Counter()
    doc_freq = Counter()
    chunk_tokens = []
    
    for idx, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Processing chunks"):
        # Use the already-processed text_for_scoring
        text = row['text_for_scoring']
        
        # Skip empty texts
        if pd.isna(text) or text.strip() == '':
            continue
        
        # Tokenize the text
        toks = tokenize(text)
        
        # Store tokens with chunk ID for reference
        chunk_tokens.append({
            'chunk_uid': row['chunk_uid'],
            'tokens': toks
        })
        
        # Update frequencies
        term_freq.update(toks)
        doc_freq.update(set(toks))  # Count each term once per chunk
    
    print(f"\n✓ Processed {len(chunk_tokens)} chunks with text")
    print(f"  Total tokens: {sum(term_freq.values()):,}")
    print(f"  Unique terms: {len(term_freq):,}")
    
    # Filter vocabulary based on minimum document frequency
    min_df = CONFIG["vocab"]["min_df"]
    max_vocab = CONFIG["vocab"]["max_vocab"]
    
    print(f"\n🔍 Filtering vocabulary...")
    print(f"  Min document frequency: {min_df}")
    print(f"  Max vocabulary size: {max_vocab}")
    
    # Get terms that appear in at least min_df chunks
    vocab_candidates = [
        (term, freq) for term, freq in term_freq.items()
        if doc_freq[term] >= min_df
    ]
    
    # Sort by frequency and take top max_vocab terms
    vocab_candidates.sort(key=lambda x: x[1], reverse=True)
    vocab_candidates = vocab_candidates[:max_vocab]
    terms = [term for term, _ in vocab_candidates]
    
    print(f"\n✓ Filtered vocabulary: {len(terms)} terms")
    
    # Show some statistics
    if len(terms) > 0:
        print(f"\n📊 Vocabulary statistics:")
        print(f"  Most common terms:")
        for term, freq in vocab_candidates[:10]:
            print(f"    - '{term}': {freq:,} occurrences (in {doc_freq[term]} chunks)")
        
        # Show filtering impact
        removed = len(term_freq) - len(terms)
        print(f"\n  Removed {removed:,} rare terms (appeared in < {min_df} chunks)")
    
    # Save vocabulary
    vocab_df = pd.DataFrame({
        'term': terms,
        'term_freq': [term_freq[t] for t in terms],
        'doc_freq': [doc_freq[t] for t in terms]
    })
    fs.save_data(vocab_df, "vocabulary", "Other_data", "csv")
    
    # Save frequencies for all terms (not just vocabulary)
    freq_data = {
        'term_freq': dict(term_freq),
        'doc_freq': dict(doc_freq),
        'n_chunks': len(chunk_tokens),
        'n_documents': chunks_df['file_path'].nunique()
    }
    fs.save_data(freq_data, "term_frequencies", "Other_data", "json")
    
    # Save the tokenized chunks for later use
    tokens_df = pd.DataFrame(chunk_tokens)
    fs.save_data(tokens_df, "chunk_tokens", "Other_data", "csv")
    
    fs.save_config("checkpoint2_vocab")
    
    print(f"\n{'='*60}")
    print("VOCABULARY BUILDING COMPLETE")
    print(f"{'='*60}")


BUILDING VOCABULARY FROM CHUNKED CORPUS
📂 Using workflow directory: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1
✓ Loading from: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1\Other_data\chunked_corpus.csv

✓ Loaded 2900 chunks from 1378 documents

📝 Tokenizing chunks...


Processing chunks: 100%|██████████| 2900/2900 [00:00<00:00, 8761.04it/s]



✓ Processed 2841 chunks with text
  Total tokens: 236,266
  Unique terms: 30,762

🔍 Filtering vocabulary...
  Min document frequency: 5
  Max vocabulary size: 50000

✓ Filtered vocabulary: 6661 terms

📊 Vocabulary statistics:
  Most common terms:
    - 'slavernij': 2,180 occurrences (in 935 chunks)
    - 'nederlandse': 1,248 occurrences (in 792 chunks)
    - 'racisme': 1,036 occurrences (in 448 chunks)
    - 'koloniale': 1,004 occurrences (in 535 chunks)
    - 'nederland': 970 occurrences (in 600 chunks)
    - 'mensen': 901 occurrences (in 569 chunks)
    - 'den': 838 occurrences (in 514 chunks)
    - 'utrecht': 799 occurrences (in 357 chunks)
    - 'onderzoek': 790 occurrences (in 509 chunks)
    - 'amsterdam': 769 occurrences (in 465 chunks)

  Removed 24,101 rare terms (appeared in < 5 chunks)
✓ Saved: Other_data/vocabulary.csv
✓ Saved: Other_data/term_frequencies.json
✓ Saved: Other_data/chunk_tokens.csv
✓ Config saved: config_checkpoint2_vocab_20251030_171524.json

VOCABULARY BUI

✅ **CHECKPOINT 2 COMPLETE** - Vocabulary built and saved

**Resume**: Load `terms`, `term_freq`, `doc_freq` from saved files

---
# CHECKPOINT 3: Dictionary Expansion
---

⚠️ **MANUAL CURATION REQUIRED AFTER THIS STEP**

Expand seed keywords using semantic similarity.

In [62]:
# ============================================================
# CELL 3.1: LOAD SENTENCE TRANSFORMER MODEL
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 3 START - LOADING VOCABULARY & MODEL")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load vocabulary
vocab_path = fs.folders['Other_data'] / 'vocabulary.csv'
if not vocab_path.exists():
    raise FileNotFoundError(f"Vocabulary not found at {vocab_path}")

vocab_df = pd.read_csv(vocab_path)
terms = vocab_df['term'].tolist()
print(f"✓ Loaded vocabulary: {len(terms)} terms")

# Load term frequencies
term_freq_path = fs.folders['Other_data'] / 'term_frequencies.json'
if term_freq_path.exists():
    with open(term_freq_path, 'r') as f:
        freq_data = json.load(f)
    term_freq = freq_data['term_freq']
    doc_freq = freq_data['doc_freq']
    print(f"✓ Loaded term frequencies")
else:
    print(f"⚠ Term frequencies not found, will create from vocabulary")
    term_freq = {row['term']: row['term_freq'] for _, row in vocab_df.iterrows() if 'term_freq' in vocab_df.columns}
    doc_freq = {row['term']: row['doc_freq'] for _, row in vocab_df.iterrows() if 'doc_freq' in vocab_df.columns}

print(f"\n{'='*60}")
print("LOADING SENTENCE TRANSFORMER MODEL")
print(f"{'='*60}")

if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
    model_path = CONFIG['paths']['pretrained_model_path']
    if Path(model_path).exists():
        st_model = SentenceTransformer(model_path)
        print(f"✓ Loaded pretrained model from: {model_path}")
    else:
        st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
        print(f"⚠ Pretrained path not found, using base model")
else:
    st_model = SentenceTransformer(CONFIG['model']['base_model_name'])
    print(f"✓ Loaded base model: {CONFIG['model']['base_model_name']}")

def st_embed(texts: list, batch_size: int = 256) -> np.ndarray:
    return st_model.encode(
        texts, 
        batch_size=batch_size, 
        show_progress_bar=False, 
        normalize_embeddings=True
    )

print(f"\n✓ Model ready")
print(f"  Max sequence length: {st_model.max_seq_length}")
print(f"  Embedding dimension: {st_model.get_sentence_embedding_dimension()}")



CHECKPOINT 3 START - LOADING VOCABULARY & MODEL


FileNotFoundError: Vocabulary not found at workflow_data\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v3\Other_data\vocabulary.csv

In [ ]:
# ============================================================
# CELL 3.2: ENCODE VOCABULARY & BUILD NN INDEX
# ============================================================

print(f"\n{'='*60}")
print("ENCODING VOCABULARY")
print(f"{'='*60}")

vocab_emb = st_embed(terms)
print(f"✓ Encoded {len(terms)} terms → shape {vocab_emb.shape}")

term2idx = {t: i for i, t in enumerate(terms)}

print("\nBuilding nearest neighbor index...")
nn = NearestNeighbors(n_neighbors=min(100, len(terms)), metric="cosine", algorithm="auto")
nn.fit(vocab_emb)
print("✓ NN index ready")

def nearest_terms(query: str, k: int = 50):
    qv = st_embed([query])[0].reshape(1, -1)
    distances, indices = nn.kneighbors(qv, n_neighbors=min(k, len(terms)))
    sims = 1.0 - distances[0]
    idxs = indices[0]
    return [(terms[i], float(sims[j])) for j, i in enumerate(idxs)]

# Save vocabulary embeddings and metadata
print(f"\n{'='*60}")
print("SAVING VOCABULARY EMBEDDINGS")
print(f"{'='*60}")

vocab_emb_path = fs.folders['Other_data'] / 'vocab_embeddings.npy'
np.save(vocab_emb_path, vocab_emb)
print(f"✓ Saved vocab embeddings: {vocab_emb.shape}")

vocab_meta = {
    'terms': terms,
    'term2idx': term2idx
}
vocab_meta_path = fs.folders['Other_data'] / 'vocab_meta.json'
with open(vocab_meta_path, 'w') as f:
    json.dump(vocab_meta, f, indent=2)
print(f"✓ Saved vocab metadata: {len(terms)} terms")

print(f"\n{'='*60}")
print("CHECKPOINT 3 COMPLETE")
print(f"{'='*60}")


In [45]:
# ============================================================
# CELL 3.3: EXPAND SEED TERMS
# ============================================================

print(f"\n{'='*60}")
print("EXPANDING SEED TERMS")
print(f"{'='*60}")

topic_rows = []
k = CONFIG['expand']['k_nearest']
min_cos = CONFIG['expand']['min_cosine']

# Use the curated/loaded topics if available, otherwise fall back to default seeds
seed_dict = CONFIG.get('topics') or CONFIG['dictionary'].get('default_topics', {})

for topic, seeds in seed_dict.items():
    print(f"  Processing: {topic}")
    seen = {}
    for s in seeds:
        # skip empty seeds
        if not s or not isinstance(s, str):
            continue
        for w, sim in nearest_terms(s, k=k):
            if doc_freq.get(w, 0) < CONFIG['vocab']['min_df']:
                continue
            if sim < min_cos:
                continue
            if w not in seen or sim > seen[w]:
                seen[w] = sim

    # ensure original seeds are kept with high score when present in vocab
    for s in seeds:
        if s in term2idx:
            seen[s] = max(seen.get(s, 0.0), 1.0)

    rows = sorted(seen.items(), key=lambda x: x[1], reverse=True)[:CONFIG['expand']['topN_per_topic']]
    for w, sc in rows:
        topic_rows.append({
            "topic": topic,
            "term": w,
            "cosine": round(sc, 4),
            "df": int(doc_freq.get(w, 0))
        })
    print(f"    → Found {len(rows)} candidate terms")

cands_df = pd.DataFrame(topic_rows)
print(f"\n✓ Generated {len(cands_df)} candidates across {cands_df['topic'].nunique()} topics")

fs.save_data(cands_df, "expanded_candidates", "Dictionary", "csv")

print(f"\n⚠️ MANUAL CURATION REQUIRED:")
print(f"  1. Review: Dictionary/expanded_candidates.csv")
print(f"  2. Remove irrelevant terms")
print(f"  3. Save as: Dictionary/curated_dictionary.csv")
print(f"  4. Then proceed to CHECKPOINT 4")

fs.save_config("checkpoint3_expansion")
print(f"\n⚠️ MANUAL CURATION REQUIRED:")
print(f"  1. Review: Dictionary/expanded_candidates.csv")
print(f"  2. Remove irrelevant terms")
print(f"  3. Save as: Dictionary/curated_dictionary.csv")
print(f"  4. Then proceed to CHECKPOINT 4")

fs.save_config("checkpoint3_expansion")
print(f"\n⚠️ MANUAL CURATION REQUIRED:")
print(f"  1. Review: Dictionary/expanded_candidates.csv")
print(f"  2. Remove irrelevant terms")
print(f"  3. Save as: Dictionary/curated_dictionary.csv")
print(f"  4. Then proceed to CHECKPOINT 4")

fs.save_config("checkpoint3_expansion")


EXPANDING SEED TERMS
  Processing: Colonialism
    → Found 300 candidate terms
  Processing: Historical Slavery
    → Found 300 candidate terms
  Processing: Modern Racism & Inequality
    → Found 300 candidate terms

✓ Generated 900 candidates across 3 topics
✓ Saved: Dictionary/expanded_candidates.csv

⚠️ MANUAL CURATION REQUIRED:
  1. Review: Dictionary/expanded_candidates.csv
  2. Remove irrelevant terms
  3. Save as: Dictionary/curated_dictionary.csv
  4. Then proceed to CHECKPOINT 4
✓ Config saved: config_checkpoint3_expansion_20251028_220739.json

⚠️ MANUAL CURATION REQUIRED:
  1. Review: Dictionary/expanded_candidates.csv
  2. Remove irrelevant terms
  3. Save as: Dictionary/curated_dictionary.csv
  4. Then proceed to CHECKPOINT 4
✓ Config saved: config_checkpoint3_expansion_20251028_220739.json

⚠️ MANUAL CURATION REQUIRED:
  1. Review: Dictionary/expanded_candidates.csv
  2. Remove irrelevant terms
  3. Save as: Dictionary/curated_dictionary.csv
  4. Then proceed to CHECKPOI

WindowsPath('workflow_data/Finetuned_Slavery-Slavery-policy_10.28.25_v1/config/config_checkpoint3_expansion_20251028_220739.json')

✅ **CHECKPOINT 3 COMPLETE** - Dictionary expanded

⚠️ **STOP HERE** - Manually curate `expanded_candidates.csv` and save as `curated_dictionary.csv`

---
# CHECKPOINT 4: Topic Vector Creation
---

Build weighted topic vectors from curated dictionary.

In [103]:
# ============================================================
# CELL 4.1: LOAD CURATED DICTIONARY & BUILD TOPIC VECTORS
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 4 START - LOADING REQUIRED DATA")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load term frequencies
term_freq_path = fs.folders['Other_data'] / 'term_frequencies.json'
if not term_freq_path.exists():
    raise FileNotFoundError(f"Term frequencies not found at {term_freq_path}")

with open(term_freq_path, 'r') as f:
    freq_data = json.load(f)
term_freq = freq_data['term_freq']
doc_freq = freq_data['doc_freq']
print(f"✓ Loaded term frequencies")

# Load vocabulary
vocab_path = fs.folders['Other_data'] / 'vocabulary.csv'
if not vocab_path.exists():
    raise FileNotFoundError(f"Vocabulary not found at {vocab_path}")

vocab_df = pd.read_csv(vocab_path)
terms = vocab_df['term'].tolist()
print(f"✓ Loaded vocabulary: {len(terms)} terms")

# Try to load vocabulary embeddings, if not exist, recreate
vocab_emb_path = fs.folders['Other_data'] / 'vocab_embeddings.npy'
vocab_meta_path = fs.folders['Other_data'] / 'vocab_meta.json'

if vocab_emb_path.exists() and vocab_meta_path.exists():
    vocab_emb = np.load(vocab_emb_path)
    with open(vocab_meta_path, 'r') as f:
        vocab_meta = json.load(f)
    term2idx = vocab_meta['term2idx']
    print(f"✓ Loaded vocab embeddings: {vocab_emb.shape}")
else:
    print(f"⚠ Vocab embeddings not found, recreating...")
    vocab_emb = st_embed(terms)
    term2idx = {t: i for i, t in enumerate(terms)}
    
    # Save for future use
    np.save(vocab_emb_path, vocab_emb)
    vocab_meta = {'terms': terms, 'term2idx': term2idx}
    with open(vocab_meta_path, 'w') as f:
        json.dump(vocab_meta, f, indent=2)
    print(f"✓ Created and saved vocab embeddings: {vocab_emb.shape}")

print(f"\n{'='*60}")
print("BUILDING TOPIC VECTORS FROM CURATED DICTIONARY")
print(f"{'='*60}")

curated_path = fs.folders['Dictionary'] / 'curated_dictionary.csv'

if not curated_path.exists():
    print(f"❌ Curated dictionary not found: {curated_path}")
    print(f"   Please complete manual curation first!")
else:
    pruned = pd.read_csv(curated_path)
    print(f"✓ Loaded curated dictionary: {len(pruned)} terms, {pruned['topic'].nunique()} topics")
    
    # Calculate SIF weights
    total_tf = max(1, sum(term_freq.values()))
    a = CONFIG['scoring']['sif_a']
    
    def term_weight(t: str) -> float:
        if not CONFIG['scoring']['use_sif']:
            return 1.0
        tf = term_freq.get(t, 1)
        return 1.0 / (a + tf / total_tf)
    
    # Build topic vectors
    topic2vec = {}
    topic2terms = defaultdict(list)
    
    for topic, sub in pruned.groupby('topic', sort=False):
        vecs, ws = [], []
        for t in sub['term']:
            if t not in term2idx:
                continue
            v = vocab_emb[term2idx[t]]
            w = term_weight(t)
            vecs.append(v)
            ws.append(w)
            topic2terms[topic].append(t)
        
        if not vecs:
            continue
        
        V = np.vstack(vecs)
        W = np.array(ws).reshape(-1, 1)
        tv = (V * W).sum(axis=0) / (W.sum() + 1e-12)
        tv = tv / (np.linalg.norm(tv) + 1e-12)
        topic2vec[topic] = tv
        print(f"  {topic}: {len(topic2terms[topic])} terms")
    
    print(f"\n✓ Created {len(topic2vec)} topic vectors")
    
    # Save using fs
    topic_vec_path = fs.folders['Other_data'] / 'topic_vectors.npy'
    np.save(topic_vec_path, topic2vec)
    print(f"✓ Saved topic vectors")
    
    meta = {
        "topics": list(topic2vec.keys()),
        "terms": dict(topic2terms)
    }
    topic_meta_path = fs.folders['Other_data'] / 'topic_vectors_meta.json'
    with open(topic_meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f"✓ Saved topic metadata")
    
    # Generate suggestions for each topic
    print(f"\n{'='*60}")
    print("GENERATING TOPIC-SPECIFIC SUGGESTIONS")
    print(f"{'='*60}")
    
    for topic in topic2vec.keys():
        print(f"  Generating suggestions for: {topic}")
        tv = topic2vec[topic]
        
        # Get similarity scores for all terms
        sims = []
        for i, term in enumerate(terms):
            te = vocab_emb[i]
            sim = float(np.dot(tv, te) / (np.linalg.norm(tv) * np.linalg.norm(te) + 1e-12))
            sims.append((term, sim))
        
        # Sort by similarity
        sims.sort(key=lambda x: x[1], reverse=True)
        
        # Take top N suggestions
        n_suggestions = CONFIG.get('expand', {}).get('k_nearest', 100)
        top_sims = sims[:n_suggestions]
        
        # Create dataframe
        out = pd.DataFrame(top_sims, columns=['term', 'similarity'])
        out.insert(0, 'topic', topic)
        out['keep'] = True
        topic_filename = f"{topic.replace(' ', '_')}_suggestions.csv"
        topic_path = fs.folders['Dictionary_suggestions'] / topic_filename
        out.to_csv(topic_path, index=False, encoding='utf-8')
    
    print(f"✓ Saved per-topic suggestions to Dictionary_suggestions/")
    
    fs.save_config("checkpoint4_vectors")



CHECKPOINT 4 START - LOADING REQUIRED DATA
✓ Loaded term frequencies
✓ Loaded vocabulary: 6661 terms
✓ Loaded vocab embeddings: (6661, 768)

BUILDING TOPIC VECTORS FROM CURATED DICTIONARY
✓ Loaded curated dictionary: 187 terms, 3 topics
  Colonialism: 50 terms
  Historical slavery: 54 terms
  Modern racism& inequality: 59 terms

✓ Created 3 topic vectors
✓ Saved topic vectors
✓ Saved topic metadata

GENERATING TOPIC-SPECIFIC SUGGESTIONS
  Generating suggestions for: Colonialism
  Generating suggestions for: Historical slavery
  Generating suggestions for: Modern racism& inequality
✓ Saved per-topic suggestions to Dictionary_suggestions/
✓ Config saved: config_checkpoint4_vectors_20251030_171636.json


✅ **CHECKPOINT 4 COMPLETE** - Topic vectors created

**Resume**: Load `topic2vec` from `Other_data/topic_vectors.npy`

---
# CHECKPOINT 5: Chunk Scoring & Confidence Classification
---

Score all chunks and classify by confidence level (High/Low/None).

In [104]:
# ============================================================
# CELL 5.1: SCORE CHUNKS
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 5 START - LOADING REQUIRED DATA")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load chunks DataFrame
chunks_path = fs.folders['Other_data'] / 'chunked_corpus.csv'
if not chunks_path.exists():
    raise FileNotFoundError(f"Chunked corpus not found at {chunks_path}")

chunks_df = pd.read_csv(chunks_path)
print(f"✓ Loaded chunks: {len(chunks_df)} chunks")

# Load topic vectors
topic_vec_path = fs.folders['Other_data'] / 'topic_vectors.npy'
topic_meta_path = fs.folders['Other_data'] / 'topic_vectors_meta.json'

if not topic_vec_path.exists() or not topic_meta_path.exists():
    raise FileNotFoundError(f"Topic vectors not found. Need both files in Other_data/")

topic2vec = np.load(topic_vec_path, allow_pickle=True).item()
with open(topic_meta_path, 'r') as f:
    topic_meta = json.load(f)

print(f"✓ Loaded topic vectors: {len(topic2vec)} topics")
print(f"  Topics: {', '.join(topic2vec.keys())}")

print(f"\n{'='*60}")
print("SCORING CHUNKS")
print(f"{'='*60}")

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# Score all chunks
records = []
for idx, chunk in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Scoring"):
    text = chunk['text_for_scoring']
    
    # Check if text is actually a valid string (not NaN, not empty, not a float)
    if isinstance(text, str) and text.strip():
        dv = st_embed([text])[0]
        row = {
            'filename': chunk['file_path'],
            'chunk_id': chunk['chunk_uid'],
            'sentence_count': chunk['sentence_count'],
            'raw_text': chunk['raw_text'],
            'text_for_scoring': text,
        }
        for topic, tv in topic2vec.items():
            row[f'cos_{topic}'] = cosine(dv, tv)
    else:
        # Handle invalid text (NaN, empty, or float values)
        row = {
            'filename': chunk['file_path'],
            'chunk_id': chunk['chunk_uid'],
            'sentence_count': chunk['sentence_count'],
            'raw_text': chunk['raw_text'],
            'text_for_scoring': '',
        }
        for topic in topic2vec.keys():
            row[f'cos_{topic}'] = 0.0
    
    records.append(row)

all_scores_df = pd.DataFrame(records)
print(f"\n✓ Scored {len(all_scores_df)} chunks across {len(topic2vec)} topics")

# Calculate metrics
topic_cols = [col for col in all_scores_df.columns if col.startswith('cos_')]
all_scores_df['max_score'] = all_scores_df[topic_cols].max(axis=1)
all_scores_df['primary_topic'] = all_scores_df[topic_cols].idxmax(axis=1).str.replace('cos_', '')

topic_scores = all_scores_df[topic_cols].values
sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]
all_scores_df['score_margin'] = sorted_scores[:, 0] - sorted_scores[:, 1]

print(f"✓ Calculated max_score, primary_topic, and score_margin")



CHECKPOINT 5 START - LOADING REQUIRED DATA
✓ Loaded chunks: 2900 chunks
✓ Loaded topic vectors: 3 topics
  Topics: Colonialism, Historical slavery, Modern racism& inequality

SCORING CHUNKS


Scoring: 100%|██████████| 2900/2900 [00:36<00:00, 79.09it/s] 



✓ Scored 2900 chunks across 3 topics
✓ Calculated max_score, primary_topic, and score_margin


In [105]:
# ============================================================
# CELL 5.2: CONFIDENCE CLASSIFICATION
# ============================================================

print(f"\n{'='*60}")
print("CONFIDENCE CLASSIFICATION")
print(f"{'='*60}")

# Thresholds
HIGH_SCORE = CONFIG['scoring']['high_confidence_score']
HIGH_MARGIN = CONFIG['scoring']['high_confidence_margin']
LOW_SCORE = CONFIG['scoring']['low_confidence_score']
LOW_MARGIN = CONFIG['scoring']['low_confidence_margin']

# Classify
high_mask = (
    (all_scores_df['max_score'] >= HIGH_SCORE) & 
    (all_scores_df['score_margin'] >= HIGH_MARGIN)
)

low_mask = (
    (all_scores_df['max_score'] >= LOW_SCORE) & 
    (all_scores_df['score_margin'] >= LOW_MARGIN) &
    ~high_mask
)

no_mask = ~(high_mask | low_mask)

high_df = all_scores_df[high_mask].copy()
low_df = all_scores_df[low_mask].copy()
no_df = all_scores_df[no_mask].copy()

high_df['confidence_level'] = 'high'
low_df['confidence_level'] = 'low'
no_df['confidence_level'] = 'none'

# Save
fs.save_data(high_df, "scores_high_confidence", "Cosine_labeling", "csv")
fs.save_data(low_df, "scores_low_confidence", "Cosine_labeling", "csv")
fs.save_data(no_df, "scores_no_confidence", "Cosine_labeling", "csv")

all_labeled = pd.concat([high_df, low_df, no_df], ignore_index=True)
fs.save_data(all_labeled, "scores_all_labeled", "Cosine_labeling", "csv")

# Report
total = len(all_scores_df)
print(f"\nTotal chunks: {total}")
print(f"\n1. HIGH CONFIDENCE: {len(high_df)} ({len(high_df)/total*100:.1f}%)")
print(f"   Mean score: {high_df['max_score'].mean():.3f}, Mean margin: {high_df['score_margin'].mean():.3f}")
print(f"\n2. LOW CONFIDENCE: {len(low_df)} ({len(low_df)/total*100:.1f}%)")
print(f"   Mean score: {low_df['max_score'].mean():.3f}, Mean margin: {low_df['score_margin'].mean():.3f}")
print(f"\n3. NO CONFIDENCE: {len(no_df)} ({len(no_df)/total*100:.1f}%)")
print(f"   Mean score: {no_df['max_score'].mean():.3f}, Mean margin: {no_df['score_margin'].mean():.3f}")

fs.save_config("checkpoint5_scoring")


CONFIDENCE CLASSIFICATION
✓ Saved: Cosine_labeling/scores_high_confidence.csv
✓ Saved: Cosine_labeling/scores_low_confidence.csv
✓ Saved: Cosine_labeling/scores_no_confidence.csv
✓ Saved: Cosine_labeling/scores_all_labeled.csv

Total chunks: 2900

1. HIGH CONFIDENCE: 69 (2.4%)
   Mean score: 0.529, Mean margin: 0.108

2. LOW CONFIDENCE: 719 (24.8%)
   Mean score: 0.443, Mean margin: 0.068

3. NO CONFIDENCE: 2112 (72.8%)
   Mean score: 0.333, Mean margin: 0.037
✓ Config saved: config_checkpoint5_scoring_20251030_171832.json


WindowsPath('workflow_data/Pretrained_Slavery-Slavery_10.30.25_v1/config/config_checkpoint5_scoring_20251030_171832.json')

✅ **CHECKPOINT 5 COMPLETE** - Chunks scored and classified

**Resume**: Load confidence CSVs from `Cosine_labeling/`

---
# CHECKPOINT 6: Training Data Preparation
---

Create train/val splits from confidence tiers.

In [106]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load the three CSV files using fs.folders
high_path = fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_path = fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_path = fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

if not high_path.exists() or not low_path.exists() or not no_path.exists():
    print(f"❌ Error: One or more score files not found")
    print(f"   Looking in: {fs.folders['Cosine_labeling']}")
    print(f"   - {high_path.name} {'✓' if high_path.exists() else '✗'}")
    print(f"   - {low_path.name} {'✓' if low_path.exists() else '✗'}")
    print(f"   - {no_path.name} {'✓' if no_path.exists() else '✗'}")
    raise FileNotFoundError("Required score files not found. Please run CHECKPOINT 5 first.")
else:
    high_df = pd.read_csv(high_path)
    low_df = pd.read_csv(low_path)
    no_df = pd.read_csv(no_path)
    
    print(f"✓ Loaded score files from: {fs.folders['Cosine_labeling']}")
    print(f"  High confidence: {len(high_df)} chunks")
    print(f"  Low confidence:  {len(low_df)} chunks")
    print(f"  No confidence:   {len(no_df)} chunks")
    print(f"  Total:           {len(high_df) + len(low_df) + len(no_df)} chunks")



CHECKPOINT 6 START - LOADING LABELED SCORE FILES
✓ Loaded score files from: workflow_data\Pretrained_Slavery-Slavery_10.30.25_v1\Cosine_labeling
  High confidence: 69 chunks
  Low confidence:  719 chunks
  No confidence:   2112 chunks
  Total:           2900 chunks


In [107]:
# ============================================================
# CELL 6.1: PREPARE LABELED DATA
# ============================================================

print(f"\n{'='*60}")
print("PREPARING LABELED DATA")
print(f"{'='*60}")

# High confidence = labeled data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['raw_text']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)
df_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(df_labeled)}")


PREPARING LABELED DATA

Label mapping:
  0: Colonialism (32 examples)
  1: Historical slavery (24 examples)
  2: Modern racism& inequality (13 examples)

Total labeled examples: 69


In [111]:
# ============================================================
# CELL 6.2: PREPARE PSEUDO-LABELED & UNLABELED DATA
# ============================================================

# =====================
# CONFIGURATION
# =====================
CONFIG = {
    # Sampling limits (balance with labeled data)
    "sampling": {
        "unlabeled_multiplier": 3,  # Max unlabeled = labeled_size * 3
        "pseudo_multiplier": 10      # Max pseudo = labeled_size * 10
    }
}

print(f"\n{'='*60}")
print("PREPARING PSEUDO-LABELED & UNLABELED DATA")
print(f"{'='*60}")

# =====================
# PREPARE PSEUDO-LABELED DATA
# =====================

# Pseudo-labeled pool (low confidence predictions)
df_pseudo = low_df.copy()
df_pseudo['text'] = df_pseudo['raw_text']
df_pseudo['label'] = df_pseudo['primary_topic']
df_pseudo['label_id'] = df_pseudo['label'].map(label2id)
df_pseudo['is_pseudo'] = True

print(f"\nPseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample pseudo-labeled data for balance
max_pseudo = len(df_labeled) * CONFIG["sampling"]["pseudo_multiplier"]
if len(df_pseudo) > max_pseudo:
    df_pseudo_sampled = df_pseudo[['text', 'label', 'label_id', 'is_pseudo']].sample(
        n=max_pseudo, random_state=42
    )
    print(f"  Sampled: {len(df_pseudo_sampled)} (to maintain balance)")
else:
    df_pseudo_sampled = df_pseudo[['text', 'label', 'label_id', 'is_pseudo']].copy()
    print(f"  Using all: {len(df_pseudo_sampled)}")

# =====================
# PREPARE UNLABELED DATA
# =====================

# Unlabeled pool (no confidence predictions)
df_unlabeled = no_df[['raw_text']].copy()
df_unlabeled.rename(columns={'raw_text': 'text'}, inplace=True)
df_unlabeled['label'] = 'UNLABELED'
df_unlabeled['label_id'] = -1
df_unlabeled['is_pseudo'] = False

# Clean: remove empty/null text
df_unlabeled = df_unlabeled[df_unlabeled['text'].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled['text'].astype(str).str.strip() != ''].copy()

print(f"\nUnlabeled pool: {len(df_unlabeled)} chunks")

# Sample unlabeled data for balance
max_unlabeled = len(df_labeled) * CONFIG["sampling"]["unlabeled_multiplier"]
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
    print(f"  Sampled: {len(df_unlabeled_sampled)} (to maintain balance)")
else:
    df_unlabeled_sampled = df_unlabeled.copy()
    print(f"  Using all: {len(df_unlabeled_sampled)}")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("DATA PREPARATION SUMMARY")
print(f"{'='*60}")
print(f"  Labeled:     {len(df_labeled)}")
print(f"  Pseudo:      {len(df_pseudo_sampled)}")
print(f"  Unlabeled:   {len(df_unlabeled_sampled)}")
print(f"  Total pool:  {len(df_labeled) + len(df_pseudo_sampled) + len(df_unlabeled_sampled)}")


PREPARING PSEUDO-LABELED & UNLABELED DATA

Pseudo-labeled pool: 719 chunks
  Sampled: 690 (to maintain balance)

Unlabeled pool: 2112 chunks
  Sampled: 207 (to maintain balance)

DATA PREPARATION SUMMARY
  Labeled:     69
  Pseudo:      690
  Unlabeled:   207
  Total pool:  966


In [112]:
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS & TRAIN/VAL SPLIT
# ============================================================

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS & TRAIN/VAL SPLIT")
print(f"{'='*60}")

# =====================
# STEP 1: GROUP DATA INTO OPTIONS FIRST
# =====================

print(f"\nStep 1: Grouping data into options...")

# =====================
# OPTION 1: LABELED ONLY
# =====================
data_opt1 = df_labeled.copy()

# =====================
# OPTION 2: LABELED + PSEUDO-LABELED
# =====================
data_opt2 = pd.concat([
    df_labeled,
    df_pseudo_sampled
], ignore_index=True)

# =====================
# OPTION 3: LABELED + UNLABELED
# =====================
data_opt3 = pd.concat([
    df_labeled,
    df_unlabeled_sampled
], ignore_index=True)

# =====================
# OPTION 4: ALL (LABELED + PSEUDO + UNLABELED)
# =====================
data_opt4 = pd.concat([
    df_labeled,
    df_pseudo_sampled,
    df_unlabeled_sampled
], ignore_index=True)

print(f"  Option 1 (Labeled only):          {len(data_opt1):>6} examples")
print(f"  Option 2 (Labeled + Pseudo):      {len(data_opt2):>6} examples")
print(f"  Option 3 (Labeled + Unlabeled):   {len(data_opt3):>6} examples")
print(f"  Option 4 (All) ⭐ RECOMMENDED:    {len(data_opt4):>6} examples")

# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# =====================

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """
    Split data into train/val, using stratification if possible.
    Only stratify on labeled data (exclude UNLABELED from stratification).
    """
    # Separate labeled and unlabeled data
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()
    
    # Check if stratified split is possible for labeled data
    if len(labeled_data) > 0:
        topic_counts = labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)
        
        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                labeled_data,
                test_size=0.2,
                stratify=labeled_data['label'],
                random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")
        else:
            train_labeled, val_labeled = train_test_split(
                labeled_data,
                test_size=0.2,
                random_state=42
            )
            print(f"  {option_name}: ⚠ Random split (some topics < 2 examples)")
        
        # Add unlabeled data to training set only (not validation)
        if len(unlabeled_data) > 0:
            train_data = pd.concat([train_labeled, unlabeled_data], ignore_index=True)
            print(f"      Added {len(unlabeled_data)} unlabeled to training")
        else:
            train_data = train_labeled
        
        val_data = val_labeled
    else:
        # Edge case: only unlabeled data (shouldn't happen but handle it)
        train_data = data
        val_data = data.head(0)  # Empty validation set
        print(f"  {option_name}: ⚠ No labeled data for validation")
    
    return train_data, val_data

# Split each option
train_opt1, val_opt1 = split_with_stratification(data_opt1, "Option 1")
train_opt2, val_opt2 = split_with_stratification(data_opt2, "Option 2")
train_opt3, val_opt3 = split_with_stratification(data_opt3, "Option 3")
train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("TRAIN/VAL SPLIT SUMMARY")
print(f"{'='*60}")
print(f"\nOption 1 (Labeled only):")
print(f"  Training:   {len(train_opt1):>6} examples")
print(f"  Validation: {len(val_opt1):>6} examples")

print(f"\nOption 2 (Labeled + Pseudo):")
print(f"  Training:   {len(train_opt2):>6} examples")
print(f"  Validation: {len(val_opt2):>6} examples")

print(f"\nOption 3 (Labeled + Unlabeled):")
print(f"  Training:   {len(train_opt3):>6} examples")
print(f"  Validation: {len(val_opt3):>6} examples")

print(f"\nOption 4 (All) ⭐ RECOMMENDED:")
print(f"  Training:   {len(train_opt4):>6} examples")
print(f"  Validation: {len(val_opt4):>6} examples")

# =====================
# SAVE DATA
# =====================

print(f"\n{'='*60}")
print("SAVING DATA")
print(f"{'='*60}")

# Save label mapping
fs.save_data(label2id, "bertje_label_mapping", "Model_finetuning", "json")
print(f"✓ Saved label mapping")

# Save all training and validation sets
fs.save_data(train_opt1, "train_data_option1_labeled_only", "Model_finetuning", "csv")
fs.save_data(val_opt1, "val_data_option1", "Model_finetuning", "csv")

fs.save_data(train_opt2, "train_data_option2_with_pseudo", "Model_finetuning", "csv")
fs.save_data(val_opt2, "val_data_option2", "Model_finetuning", "csv")

fs.save_data(train_opt3, "train_data_option3_with_unlabeled", "Model_finetuning", "csv")
fs.save_data(val_opt3, "val_data_option3", "Model_finetuning", "csv")

fs.save_data(train_opt4, "train_data_option4_all", "Model_finetuning", "csv")
fs.save_data(val_opt4, "val_data_option4", "Model_finetuning", "csv")

print(f"✓ Saved all training and validation sets")

# Save checkpoint
fs.save_config("checkpoint6_training_prep")
print(f"✓ Checkpoint saved")

print(f"\n{'='*60}")
print("✓ DATA PREPARATION COMPLETE")
print(f"{'='*60}")
print(f"\nReady for model training!")
print(f"Choose one of the training options (1-4) for your model.")


CREATING DATASET OPTIONS & TRAIN/VAL SPLIT

Step 1: Grouping data into options...
  Option 1 (Labeled only):              69 examples
  Option 2 (Labeled + Pseudo):         759 examples
  Option 3 (Labeled + Unlabeled):      276 examples
  Option 4 (All) ⭐ RECOMMENDED:       966 examples

Step 2: Splitting each option into train/val...
  Option 1: ✓ Stratified split
  Option 2: ✓ Stratified split
  Option 3: ✓ Stratified split
      Added 207 unlabeled to training
  Option 4: ✓ Stratified split
      Added 207 unlabeled to training

TRAIN/VAL SPLIT SUMMARY

Option 1 (Labeled only):
  Training:       55 examples
  Validation:     14 examples

Option 2 (Labeled + Pseudo):
  Training:      607 examples
  Validation:    152 examples

Option 3 (Labeled + Unlabeled):
  Training:      262 examples
  Validation:     14 examples

Option 4 (All) ⭐ RECOMMENDED:
  Training:      814 examples
  Validation:    152 examples

SAVING DATA
✓ Saved: Model_finetuning/bertje_label_mapping.json
✓ Saved lab

✅ **CHECKPOINT 6 COMPLETE** - Training data prepared

**Resume**: Load train/val CSVs from `Model_finetuning/`

---
# CHECKPOINT 7: Model Training (BERTJE)
---

Fine-tune Dutch BERT on labeled data.

⚠️ **Note**: This requires `transformers` library and GPU for efficient training.

In [117]:
# ============================================================
# CELL 7.1: SETUP TRAINING
# ============================================================

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding
    )
    from datasets import Dataset
    import torch
    
    print("✓ Transformers library available")
    
    # Check GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    
    TRAINING_AVAILABLE = True
    
except ImportError as e:
    print("⚠ Transformers library not available")
    print("  Install: pip install transformers datasets torch")
    TRAINING_AVAILABLE = False

✓ Transformers library available

Device: cuda
  GPU: NVIDIA GeForce RTX 3050


In [127]:
# ============================================================
# CELL 7.2: LOAD & PREPARE DATA  (adds cos_* columns for soft labels)
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("PREPARING TRAINING DATA")
    print(f"{'='*60}")
    
    # -----------------------------[ 1) PICK DATASET ]-----------------------------
    dataset_option = CONFIG['training']['dataset_option']
    if dataset_option == 'option1':
        train_dataset = train_opt1
        val_dataset   = val_opt1
    elif dataset_option == 'option2':
        train_dataset = train_opt2
        val_dataset   = val_opt2
    elif dataset_option == 'option3':
        train_dataset = train_opt3
        val_dataset   = val_opt3
    else:
        train_dataset = train_opt4
        val_dataset   = val_opt4
    
    print(f"\nUsing {dataset_option}: {len(train_dataset)} examples")
    
    # -----------------------------[ 2) LOAD MODEL + TOKENIZER ]-------------------
    from pathlib import Path
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
    
    if CONFIG['model']['use_pretrained'] and CONFIG['paths']['pretrained_model_path']:
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            model_name = model_path
            print(f"✓ Loading pretrained model from: {model_path}")
        else:
            model_name = "GroNLP/bert-base-dutch-cased"
            print(f"⚠ Pretrained path not found, using base model")
    else:
        model_name = "GroNLP/bert-base-dutch-cased"
        print(f"✓ Using base model: GroNLP/bert-base-dutch-cased")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id
    )
    model.to(device)
    print(f"\n✓ Model loaded: {model_name}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # -----------------------------[ 3) TOKENIZER FN ]-----------------------------
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding=False,
            truncation=True,
            max_length=512
        )
    
    # -----------------------------[ 4) PICK COSINE COLUMNS ]----------------------
    # Auto-detect columns that start with 'cos_' or use CONFIG['cosine']['columns']
    import numpy as np
    from datasets import Dataset
    
    train_cols = list(train_dataset.columns)
    default_cos_cols = [c for c in train_cols if isinstance(c, str) and c.startswith("cos_")]
    cos_cfg = CONFIG.get('cosine', {}) if isinstance(CONFIG.get('cosine', {}), dict) else {}
    cos_cols = cos_cfg.get('columns', default_cos_cols)
    
    # Keep only numeric cosine columns that exist in both splits
    cos_cols = [c for c in cos_cols if c in train_dataset.columns and c in val_dataset.columns]
    # Optional: ensure they are numeric
    for c in list(cos_cols):
        try:
            _ = np.asarray(train_dataset[c].astype(float))
            _ = np.asarray(val_dataset[c].astype(float))
        except Exception:
            print(f"  ⚠ Skipping non-numeric cosine column: {c}")
            cos_cols.remove(c)
    
    if len(cos_cols) == 0:
        print("ℹ No cos_* columns found. Training will fall back to hard labels.")
    else:
        print(f"✓ Cosine columns detected for soft labels: {cos_cols}")
    
    # -----------------------------[ 5) BUILD HF DATASETS ]------------------------
    # Include labels and all detected cos_* columns alongside tokenized inputs
    # Note: do not drop cos_* when mapping or set_format is applied
    base_train_cols = ['text', 'label_id'] + cos_cols
    base_val_cols   = ['text', 'label_id'] + cos_cols
    
    # guard against missing columns due to prior filtering
    base_train_cols = [c for c in base_train_cols if c in train_dataset.columns]
    base_val_cols   = [c for c in base_val_cols   if c in val_dataset.columns]
    
    train_labeled = train_dataset[train_dataset['label_id'] != -1].copy()
    hf_train = Dataset.from_pandas(train_labeled[base_train_cols].reset_index(drop=True))
    hf_val   = Dataset.from_pandas(val_dataset[base_val_cols].reset_index(drop=True))
    
    # map tokenizer (do not remove cosine columns)
    hf_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_val   = hf_val.map(  tokenize_function, batched=True, remove_columns=['text'])
    
    # rename label column
    hf_train = hf_train.rename_column('label_id', 'labels')
    hf_val   = hf_val.rename_column('label_id', 'labels')
    
    # set tensor format, include cos_* so collator can see them
    tensor_cols_train = ['input_ids', 'attention_mask', 'labels'] + cos_cols
    tensor_cols_val   = ['input_ids', 'attention_mask', 'labels'] + cos_cols
    hf_train.set_format(type='torch', columns=tensor_cols_train)
    hf_val.set_format(type='torch',   columns=tensor_cols_val)
    
    # data collator stays the same here, CELL 7.3 will wrap it when needed
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    print(f"\n✓ Data prepared")
    print(f"  Train: {len(hf_train)}, Val: {len(hf_val)}")
    if len(cos_cols) > 0:
        print(f"  Included cosine columns in HF datasets: {cos_cols}")
    else:
        print(f"  No cosine columns included (none detected)")



PREPARING TRAINING DATA

Using option2: 607 examples
✓ Using base model: GroNLP/bert-base-dutch-cased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✓ Model loaded: GroNLP/bert-base-dutch-cased
  Parameters: 109,139,715
✓ Cosine columns detected for soft labels: ['cos_Colonialism', 'cos_Historical slavery', 'cos_Modern racism& inequality']


Map:   0%|          | 0/607 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]


✓ Data prepared
  Train: 607, Val: 152
  Included cosine columns in HF datasets: ['cos_Colonialism', 'cos_Historical slavery', 'cos_Modern racism& inequality']


In [128]:
# ============================================================
# CELL 7.3: TRAIN MODEL  (BERTje with cosine soft labels + reject gate)
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING MODEL  (BERTje with cosine soft labels + reject gate)")
    print(f"{'='*60}")

    # -----------------------------[ IMPORTS ]-----------------------------
    import os, json
    import numpy as np
    import torch
    import torch.nn.functional as F
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from transformers import TrainingArguments, Trainer
    from transformers import DataCollatorWithPadding

    # -----------------------------[ 0) METRICS: unchanged ]----------------
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    # -----------------------------[ 1) COSINE SOFT-LABEL SETUP ]----------
    print("\n[SETUP] Detecting cosine columns and soft-label settings...")
    num_labels = int(getattr(model.config, "num_labels", 3))

    train_cols = hf_train.column_names
    default_cos_cols = [c for c in train_cols if c.startswith("cos_")]
    cos_cfg = CONFIG.get('cosine', {}) if isinstance(CONFIG.get('cosine', {}), dict) else {}
    cos_cols = cos_cfg.get('columns', default_cos_cols)

    if len(cos_cols) != num_labels:
        print(f"  ⚠ Expected {num_labels} cosine columns, found {len(cos_cols)}: {cos_cols}. Using hard labels.")
        cos_cols = []  # disable soft-label path

    cos_tau = float(cos_cfg.get('softmax_tau', 0.5))         # temperature for soft labels from cosine
    other_threshold = float(cos_cfg.get('other_threshold', 0.40))  # low max-cos => likely irrelevant
    other_weight = float(cos_cfg.get('other_weight', 0.30))        # weight floor for likely-irrelevant

    # -----------------------------[ 2) DATA COLLATOR (passes cos_*) ]------
    if cos_cols:
        base_collator = data_collator if data_collator is not None else DataLakollocatorWithPadding(tokenizer)

        class CollatorWithCosine:
            def __init__(self, base, cos_columns):
                self.base = base
                self.cos_columns = list(cos_columns)
            def __call__(self, features):
                # keep cosine values before base collator filters anything
                cos_buf = {c: [float(f.get(c, 0.0)) for f in features] for c in self.cos_columns}
                batch = self.base(features)  # turns text parts into tensors
                # add cosine tensors
                for c, vals in cos_buf.items():
                    batch[c] = torch.tensor(vals, dtype=torch.float)
                return batch

        effective_collator = CollatorWithCosine(base_collator, cos_cols)
        print(f"  ✓ Using soft labels from cosine columns: {cos_cols}")
        print(f"  ✓ Cosine softmax tau: {cos_tau}, irrelevant gate during training (thr={other_threshold}, floor={other_weight})")
    else:
        effective_collator = data_collator
        print("  ✓ Proceeding with hard-label training (no cosine columns used)")

    # -----------------------------[ 3) CUSTOM TRAINER (fix: accept num_items_in_batch) ]---
    class SoftLabelTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            # strip out non-model keys
            model_inputs = {k: v for k, v in inputs.items() if k not in (["labels"] + list(cos_cols))}
            outputs = model(**model_inputs)  # standard forward
            logits = outputs.logits  # [B, C]

            if cos_cols:
                cos_stack = torch.stack([inputs[c].float() for c in cos_cols], dim=1)  # [B, C]
                # temperature softmax on cosine scores -> soft targets
                cos_norm = (cos_stack / max(cos_tau, 1e-6)).softmax(dim=1)            # [B, C]
                # per-example weights (downweight likely-irrelevant)
                w = cos_stack.max(dim=1).values
                w = torch.where(w < other_threshold, torch.full_like(w, other_weight), w)
                w = torch.clamp(w, min=1e-3)
                # cross-entropy with soft labels
                logp = F.log_softmax(logits, dim=1)                                   # [B, C]
                loss_vec = -(cos_norm * logp).sum(dim=1)                               # [B]
                loss = (loss_vec * w).sum() / w.sum()
            else:
                loss = F.cross_entropy(logits, labels)

            return (loss, outputs) if return_outputs else loss

    # -----------------------------[ 4) TRAINING ARGS: add remove_unused_columns=False ]----
    model_output_dir = str(fs.folders['Model_finetuning'])

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=CONFIG['training']['num_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size_train'],
        per_device_eval_batch_size=CONFIG['training']['batch_size_eval'],
        learning_rate=CONFIG['training']['learning_rate'],
        weight_decay=CONFIG['training']['weight_decay'],
        warmup_ratio=CONFIG['training']['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=str(fs.folders['Model_finetuning'] / "logs"),
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
        remove_unused_columns=False,  # <<< IMPORTANT so cos_* reach the collator
    )

    # -----------------------------[ 5) TRAIN ]-----------------------------
    trainer = SoftLabelTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=effective_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\nStarting training for {CONFIG['training']['num_epochs']} epochs...")
    train_result = trainer.train()

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")

    # -----------------------------[ 6) SAVE MODEL + TOKENIZER: unchanged ]-
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)

    # -----------------------------[ 7) EVALUATE: unchanged ]---------------
    eval_results = trainer.evaluate()

    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")

    # -----------------------------[ 8) NEW: save soft-label info ]---------
    soft_info = {
        "used_cosine_columns": cos_cols,
        "cosine_softmax_tau": cos_tau,
        "other_threshold": other_threshold,
        "other_weight_floor": other_weight,
        "num_labels": int(num_labels),
        "used_soft_labels": bool(cos_cols)
    }
    fs.save_data(soft_info, "softlabel_training_info", "Model_finetuning", "json")
    print("✓ Saved: Model_finetuning/softlabel_training_info.json")

    # -----------------------------[ 9) NEW: calibrate reject tiers ]-------
    print("\n[CALIBRATION] Fitting temperature and computing tier thresholds...")
    val_pred = trainer.predict(hf_val)
    val_logits = val_pred.predictions
    val_labels = np.array(hf_val["labels"])

    def nll_with_T(T):
        T = max(0.05, float(T))
        z = torch.tensor(val_logits) / T
        logp = torch.log_softmax(z, dim=1).numpy()
        return -float(np.mean([logp[i, val_labels[i]] for i in range(len(val_labels))]))

    T = 1.0
    for _ in range(20):  # cheap line search
        cands = [max(0.05, T * f) for f in (0.5, 0.75, 1.0, 1.25, 1.5)]
        losses = [nll_with_T(c) for c in cands]
        T = cands[int(np.argmin(losses))]

    p_cal = torch.softmax(torch.tensor(val_logits) / T, dim=1).numpy()
    pmax_cal = p_cal.max(axis=1)

    # Target shares for tiers (simple defaults; override in CONFIG['cosine'] if you want)
    hi_share  = float(cos_cfg.get('target_high_share', 0.30))
    med_share = float(cos_cfg.get('target_med_share', 0.30))

    hi_tau  = float(np.quantile(pmax_cal, 1.0 - min(max(hi_share, 0.01), 0.95)))
    med_tau = float(np.quantile(pmax_cal, 1.0 - min(max(hi_share + med_share, 0.02), 0.98)))

    calib = {
        "temperature_T": T,
        "tier_thresholds": {
            "HIGH":   hi_tau,
            "MEDIUM": med_tau,
            "LOW_or_IRRELEVANT": 0.0
        },
        "note": "At inference: softmax(logits / T). If pmax≥HIGH→HIGH, if MEDIUM≤pmax<HIGH→MEDIUM, else LOW/Irrelevant."
    }
    fs.save_data(calib, "bert_temperature_and_tiers", "Model_finetuning", "json")
    print("✓ Saved: Model_finetuning/bert_temperature_and_tiers.json")

    # -----------------------------[ 10) SAVE METRICS + CHECKPOINT: same ]--
    metrics = {
        "train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": CONFIG['training']['num_epochs'],
        "dataset_used": dataset_option,
    }
    fs.save_data(metrics, "training_metrics", "Model_finetuning", "json")
    fs.save_config("checkpoint7_trained")

    print(f"\n✓ Model and metrics saved")
    print("✓ Soft labels used" if cos_cols else "✓ Hard labels used (no cosine columns detected)")

else:
    print("⚠ Skipping training - transformers library not available")



TRAINING MODEL  (BERTje with cosine soft labels + reject gate)

[SETUP] Detecting cosine columns and soft-label settings...
  ✓ Using soft labels from cosine columns: ['cos_Colonialism', 'cos_Historical slavery', 'cos_Modern racism& inequality']
  ✓ Cosine softmax tau: 0.5, irrelevant gate during training (thr=0.4, floor=0.3)

Starting training for 5 epochs...


  0%|          | 0/190 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': nan, 'eval_accuracy': 0.2894736842105263, 'eval_precision': 0.30240252468695916, 'eval_recall': 0.2894736842105263, 'eval_f1': 0.175854108956602, 'eval_runtime': 5.3526, 'eval_samples_per_second': 28.397, 'eval_steps_per_second': 0.934, 'epoch': 1.0}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 0.0, 'epoch': 1.32}


  0%|          | 0/5 [00:00<?, ?it/s]

...

{'eval_loss': nan, 'eval_accuracy': 0.2894736842105263, 'eval_precision': 0.30240252468695916, 'eval_recall': 0.2894736842105263, 'eval_f1': 0.175854108956602, 'eval_runtime': 4.9896, 'eval_samples_per_second': 30.464, 'eval_steps_per_second': 1.002, 'epoch': 5.0}
{'train_runtime': 550.8827, 'train_samples_per_second': 5.509, 'train_steps_per_second': 0.345, 'train_loss': 0.0, 'epoch': 5.0}

TRAINING COMPLETE


  0%|          | 0/5 [00:00<?, ?it/s]


Validation Results:
  Accuracy:  0.2895
  Precision: 0.3024
  Recall:    0.2895
  F1 Score:  0.1759
✓ Saved: Model_finetuning/softlabel_training_info.json
✓ Saved: Model_finetuning/softlabel_training_info.json

[CALIBRATION] Fitting temperature and computing tier thresholds...


  0%|          | 0/5 [00:00<?, ?it/s]

✓ Saved: Model_finetuning/bert_temperature_and_tiers.json
✓ Saved: Model_finetuning/bert_temperature_and_tiers.json
✓ Saved: Model_finetuning/training_metrics.json
✓ Config saved: config_checkpoint7_trained_20251030_182737.json

✓ Model and metrics saved
✓ Soft labels used


In [22]:
# ============================================================
# CELL 7.3: TRAIN MODEL
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING MODEL")
    print(f"{'='*60}")
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    # Training arguments
    model_output_dir = str(fs.folders['Model_finetuning'])
    
    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=CONFIG['training']['num_epochs'],
        per_device_train_batch_size=CONFIG['training']['batch_size_train'],
        per_device_eval_batch_size=CONFIG['training']['batch_size_eval'],
        learning_rate=CONFIG['training']['learning_rate'],
        weight_decay=CONFIG['training']['weight_decay'],
        warmup_ratio=CONFIG['training']['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=str(fs.folders['Model_finetuning'] / "logs"),
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    print(f"\nStarting training for {CONFIG['training']['num_epochs']} epochs...")
    train_result = trainer.train()
    
    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    
    # Save model
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)
    
    # Evaluate
    eval_results = trainer.evaluate()
    
    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")
    
    # Save metrics
    metrics = {
        "train_loss": train_result.training_loss,
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": CONFIG['training']['num_epochs'],
        "dataset_used": dataset_option,
    }
    
    fs.save_data(metrics, "training_metrics", "Model_finetuning", "json")
    fs.save_config("checkpoint7_trained")
    
    print(f"\n✓ Model and metrics saved")
else:
    print("⚠ Skipping training - transformers library not available")


TRAINING MODEL

Starting training for 3 epochs...


  0%|          | 0/996 [00:00<?, ?it/s]

{'loss': 0.4401, 'grad_norm': 11.133905410766602, 'learning_rate': 9.800000000000001e-06, 'epoch': 0.15}
{'loss': 0.2769, 'grad_norm': 7.9429707527160645, 'learning_rate': 1.9600000000000002e-05, 'epoch': 0.3}
{'loss': 0.3182, 'grad_norm': 17.560598373413086, 'learning_rate': 1.8950892857142858e-05, 'epoch': 0.45}
{'loss': 0.2684, 'grad_norm': 23.28594398498535, 'learning_rate': 1.783482142857143e-05, 'epoch': 0.6}
{'loss': 0.3301, 'grad_norm': 11.638723373413086, 'learning_rate': 1.671875e-05, 'epoch': 0.75}
{'loss': 0.2435, 'grad_norm': 29.42177391052246, 'learning_rate': 1.5602678571428574e-05, 'epoch': 0.9}


  0%|          | 0/42 [00:00<?, ?it/s]

{'eval_loss': 0.22826729714870453, 'eval_accuracy': 0.9125188536953243, 'eval_precision': 0.9173267109504935, 'eval_recall': 0.9125188536953243, 'eval_f1': 0.9130577753926096, 'eval_runtime': 11.3351, 'eval_samples_per_second': 116.981, 'eval_steps_per_second': 3.705, 'epoch': 1.0}
{'loss': 0.2093, 'grad_norm': 54.80517578125, 'learning_rate': 1.4486607142857143e-05, 'epoch': 1.05}
{'loss': 0.1379, 'grad_norm': 28.29879379272461, 'learning_rate': 1.3370535714285714e-05, 'epoch': 1.2}
{'loss': 0.2204, 'grad_norm': 4.803442001342773, 'learning_rate': 1.2254464285714287e-05, 'epoch': 1.36}
{'loss': 0.1828, 'grad_norm': 5.236793041229248, 'learning_rate': 1.113839285714286e-05, 'epoch': 1.51}
{'loss': 0.1246, 'grad_norm': 0.43263348937034607, 'learning_rate': 1.0022321428571429e-05, 'epoch': 1.66}
{'loss': 0.1378, 'grad_norm': 13.823603630065918, 'learning_rate': 8.906250000000001e-06, 'epoch': 1.81}
{'loss': 0.1765, 'grad_norm': 6.028990745544434, 'learning_rate': 7.790178571428572e-06, '

...

{'eval_loss': 0.21923618018627167, 'eval_accuracy': 0.9215686274509803, 'eval_precision': 0.9217974529218909, 'eval_recall': 0.9215686274509803, 'eval_f1': 0.9206193430472789, 'eval_runtime': 11.2005, 'eval_samples_per_second': 118.388, 'eval_steps_per_second': 3.75, 'epoch': 2.0}
{'loss': 0.0933, 'grad_norm': 2.015324115753174, 'learning_rate': 6.674107142857143e-06, 'epoch': 2.11}
{'loss': 0.0775, 'grad_norm': 0.18117284774780273, 'learning_rate': 5.558035714285714e-06, 'epoch': 2.26}
{'loss': 0.0724, 'grad_norm': 31.171051025390625, 'learning_rate': 4.441964285714286e-06, 'epoch': 2.41}
{'loss': 0.0637, 'grad_norm': 0.0270305797457695, 'learning_rate': 3.3258928571428572e-06, 'epoch': 2.56}
{'loss': 0.0477, 'grad_norm': 46.910865783691406, 'learning_rate': 2.2321428571428573e-06, 'epoch': 2.71}
{'loss': 0.0619, 'grad_norm': 10.829289436340332, 'learning_rate': 1.1160714285714287e-06, 'epoch': 2.86}


  0%|          | 0/42 [00:00<?, ?it/s]

{'eval_loss': 0.2921321988105774, 'eval_accuracy': 0.9230769230769231, 'eval_precision': 0.9227142276022363, 'eval_recall': 0.9230769230769231, 'eval_f1': 0.9225774749942328, 'eval_runtime': 11.1783, 'eval_samples_per_second': 118.622, 'eval_steps_per_second': 3.757, 'epoch': 3.0}
{'train_runtime': 556.5764, 'train_samples_per_second': 28.589, 'train_steps_per_second': 1.79, 'train_loss': 0.17809077439059215, 'epoch': 3.0}

TRAINING COMPLETE


  0%|          | 0/42 [00:00<?, ?it/s]


Validation Results:
  Accuracy:  0.9231
  Precision: 0.9227
  Recall:    0.9231
  F1 Score:  0.9226
✓ Saved: Model_finetuning/training_metrics.json
✓ Config saved: config_checkpoint7_trained_20251030_150714.json

✓ Model and metrics saved


✅ **CHECKPOINT 7 COMPLETE** - Model trained and saved

**Resume**: Load model from `Model_finetuning/`

---
# CHECKPOINT 8: BERTJE Labeling
---

Use trained BERTJE model (or base model) to label entire corpus.

**Options**:
- Use trained model from CHECKPOINT 7 (recommended)
- Use base model if training was skipped (poor results)

**Outputs**: `Bertje_labeling/` with confidence-classified predictions

In [78]:
# ============================================================
# CELL 8.1: SETUP
# ============================================================

# Check if transformers is available
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    BERT_AVAILABLE = True
    print("✓ Transformers library available")
except ImportError:
    print("⚠ Transformers not available")
    print("  Install: pip install transformers torch")
    BERT_AVAILABLE = False

✓ Transformers library available


In [79]:
# ============================================================
# CELL 8.2: LOAD MODEL (TRAINED OR BASE)
# ============================================================

if BERT_AVAILABLE:
    print(f"\n{'='*60}")
    print("LOADING MODEL FOR LABELING")
    print(f"{'='*60}")
    
    USE_TRAINED_MODEL = True  # Set False to use base (untrained) model
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    if USE_TRAINED_MODEL:
        model_path = str(fs.folders['Model_finetuning'])
        if not (Path(model_path) / 'config.json').exists():
            print(f"⚠ No trained model found, using base model")
            USE_TRAINED_MODEL = False
        else:
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            bertje_model = AutoModelForSequenceClassification.from_pretrained(model_path)
            id2label_bert = bertje_model.config.id2label
            label2id_bert = bertje_model.config.label2id
            print(f"✓ Loaded trained model from: {model_path}")
    
    if not USE_TRAINED_MODEL:
        base_name = "GroNLP/bert-base-dutch-cased"
        tokenizer = AutoTokenizer.from_pretrained(base_name)
        bertje_model = AutoModelForSequenceClassification.from_pretrained(
            base_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
        )
        id2label_bert = id2label
        label2id_bert = label2id
        print(f"✓ Loaded base model (untrained): {base_name}")
        print(f"  ⚠ Results will be random without training!")
    
    bertje_model.to(device)
    bertje_model.eval()
    print(f"\n✓ Model ready | Labels: {list(label2id_bert.keys())}")


LOADING MODEL FOR LABELING
Device: cuda
✓ Loaded trained model from: workflow_data\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v2\Model_finetuning

✓ Model ready | Labels: ['Colonialism', 'Historical_Slavery', 'Modern_Racism_Inequality']


In [80]:
# ============================================================
# CELL 8.3: PREPARE CORPUS
# ============================================================

if BERT_AVAILABLE:
    # Load scored data if available, else use chunks
    scored_path = fs.folders['Cosine_labeling'] / 'scores_all_labeled.csv'
    if scored_path.exists():
        corpus_for_bert = pd.read_csv(scored_path)
        source = 'cosine scoring'
    else:
        corpus_for_bert = chunks_df.copy()
        source = 'chunks'
    
    # Ensure text column
    if 'raw_text' in corpus_for_bert.columns:
        corpus_for_bert['text'] = corpus_for_bert['raw_text']
    elif 'text_for_scoring' in corpus_for_bert.columns:
        corpus_for_bert['text'] = corpus_for_bert['text_for_scoring']
    
    # Filter empties
    corpus_for_bert = corpus_for_bert[
        corpus_for_bert['text'].notna() & (corpus_for_bert['text'].str.len() > 0)
    ].copy()
    
    print(f"✓ Loaded {len(corpus_for_bert)} chunks from {source}")

✓ Loaded 7828 chunks from cosine scoring


In [81]:
# ============================================================
# CELL 8.4: RUN PREDICTIONS
# ============================================================

if BERT_AVAILABLE:
    print(f"\n{'='*60}")
    print("RUNNING BERTJE PREDICTIONS")
    print(f"{'='*60}")
    
    def predict_bert_batch(texts, batch_size=32):
        preds, probs_list, confs = [], [], []
        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc="Labeling"):
                batch = texts[i:i+batch_size]
                inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                outputs = bertje_model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                confidence, predicted = torch.max(probs, dim=-1)
                preds.extend(predicted.cpu().numpy())
                probs_list.extend(probs.cpu().numpy())
                confs.extend(confidence.cpu().numpy())
        return preds, probs_list, confs
    
    texts = corpus_for_bert['text'].tolist()
    predictions, probabilities, confidences = predict_bert_batch(texts)
    
    # Add results
    corpus_for_bert['bert_label'] = [id2label_bert[p] for p in predictions]
    corpus_for_bert['bert_confidence'] = confidences
    prob_arr = np.array(probabilities)
    sorted_p = np.sort(prob_arr, axis=1)[:, ::-1]
    corpus_for_bert['bert_margin'] = sorted_p[:, 0] - sorted_p[:, 1]
    
    print(f"\n✓ Labeled {len(corpus_for_bert)} chunks")
    print(f"\nLabel distribution:")
    print(corpus_for_bert['bert_label'].value_counts())
    print(f"\nConfidence: mean={corpus_for_bert['bert_confidence'].mean():.3f}, median={corpus_for_bert['bert_confidence'].median():.3f}")


RUNNING BERTJE PREDICTIONS


Labeling: 100%|██████████| 245/245 [03:51<00:00,  1.06it/s]


✓ Labeled 7828 chunks

Label distribution:
bert_label
Colonialism                 3921
Modern_Racism_Inequality    3395
Historical_Slavery           512
Name: count, dtype: int64

Confidence: mean=0.970, median=0.999


In [82]:
# ============================================================
# CELL 8.5: CLASSIFY & SAVE
# ============================================================

if BERT_AVAILABLE:
    # Classify confidence tiers
    def classify_tier(row):
        c, m = row['bert_confidence'], row['bert_margin']
        if c >= 0.80 and m >= 0.20: return 'high'
        if c >= 0.60 and m >= 0.10: return 'medium'
        if c >= 0.40 and m >= 0.05: return 'low'
        return 'very_low'
    
    corpus_for_bert['bert_tier'] = corpus_for_bert.apply(classify_tier, axis=1)
    
    print(f"\nTier distribution:")
    for tier in ['high', 'medium', 'low', 'very_low']:
        cnt = (corpus_for_bert['bert_tier'] == tier).sum()
        print(f"  {tier.upper():10s}: {cnt} ({cnt/len(corpus_for_bert)*100:.1f}%)")
    
    # Save
    fs.save_data(corpus_for_bert, "bert_labeled_all", "Bertje_labeling", "csv")
    for tier in ['high', 'medium', 'low', 'very_low']:
        tier_df = corpus_for_bert[corpus_for_bert['bert_tier'] == tier]
        if len(tier_df) > 0:
            fs.save_data(tier_df, f"bert_labeled_{tier}", "Bertje_labeling", "csv")
    
    # Compare with cosine if available
    if 'primary_topic' in corpus_for_bert.columns:
        agree = (corpus_for_bert['bert_label'] == corpus_for_bert['primary_topic']).sum()
        pct = agree/len(corpus_for_bert)*100
        print(f"\nBERT vs Cosine: {agree} agree ({pct:.1f}%), {len(corpus_for_bert)-agree} differ")
        disagree = corpus_for_bert[corpus_for_bert['bert_label'] != corpus_for_bert['primary_topic']]
        if len(disagree) > 0:
            fs.save_data(disagree, "bert_vs_cosine_DIFF", "Bertje_labeling", "csv")
    
    fs.save_config("checkpoint8_bertje")
    print(f"\n✓ Saved to Bertje_labeling/")


Tier distribution:
  HIGH      : 7381 (94.3%)
  MEDIUM    : 303 (3.9%)
  LOW       : 107 (1.4%)
  VERY_LOW  : 37 (0.5%)
✓ Saved: Bertje_labeling/bert_labeled_all.csv
✓ Saved: Bertje_labeling/bert_labeled_high.csv
✓ Saved: Bertje_labeling/bert_labeled_medium.csv
✓ Saved: Bertje_labeling/bert_labeled_low.csv
✓ Saved: Bertje_labeling/bert_labeled_very_low.csv

BERT vs Cosine: 6530 agree (83.4%), 1298 differ
✓ Saved: Bertje_labeling/bert_vs_cosine_DIFF.csv
✓ Config saved: config_checkpoint8_bertje_20251030_164725.json

✓ Saved to Bertje_labeling/


In [83]:
# Load the data
cosine_no_conf = pd.read_csv('Cosine_labeling/scores_no_confidence.csv')
bert_all = pd.read_csv('Bertje_labeling/bert_labeled_all.csv')

# Merge to see what BERTje labeled the NO CONFIDENCE chunks
merged = cosine_no_conf.merge(bert_all, on='chunk_id' or 'text')

print(merged['bert_label'].value_counts())
print(f"Mean confidence on NO CONF chunks: {merged['bert_confidence'].mean():.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'Cosine_labeling/scores_no_confidence.csv'

In [88]:
# Load data
cosine_no = pd.read_csv('C:\\Users\\Home\\policy-analysis\\workflow_data\\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v2\\Cosine_labeling\\scores_no_confidence.csv')
bert_all = pd.read_csv('C:\\Users\\Home\\policy-analysis\\workflow_data\\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v2\\Bertje_labeling\\bert_labeled_all.csv')

# Find NO CONFIDENCE chunks in BERTje results
# (merge on whatever ID column you have, or 'text' if no ID)
no_conf_chunks = cosine_no['chunk_id'].values  # or chunk_id
bert_on_no_conf = bert_all[bert_all['chunk_id'].isin(no_conf_chunks)]

print("BERTje labels on Cosine NO CONFIDENCE chunks:")
print(bert_on_no_conf['bert_label'].value_counts())
print(f"\nMean confidence: {bert_on_no_conf['bert_confidence'].mean():.3f}")
print(f"Mean margin: {bert_on_no_conf['bert_margin'].mean():.3f}")

# Compare to HIGH CONFIDENCE chunks
cosine_high = pd.read_csv('C:\\Users\\Home\\policy-analysis\\workflow_data\\Finetuned_Slavery-policy-Slavery-policy_10.30.25_v2\\Cosine_labeling\\scores_high_confidence.csv')
high_conf_chunks = cosine_high['chunk_id'].values
bert_on_high = bert_all[bert_all['chunk_id'].isin(high_conf_chunks)]

print("\n" + "="*60)
print("BERTje labels on Cosine HIGH CONFIDENCE chunks:")
print(bert_on_high['bert_label'].value_counts())
print(f"\nMean confidence: {bert_on_high['bert_confidence'].mean():.3f}")
print(f"Mean margin: {bert_on_high['bert_margin'].mean():.3f}")

BERTje labels on Cosine NO CONFIDENCE chunks:
bert_label
Colonialism                 2818
Historical_Slavery           441
Modern_Racism_Inequality     391
Name: count, dtype: int64

Mean confidence: 0.967
Mean margin: 0.935

BERTje labels on Cosine HIGH CONFIDENCE chunks:
bert_label
Modern_Racism_Inequality    2021
Colonialism                   57
Historical_Slavery             7
Name: count, dtype: int64

Mean confidence: 0.993
Mean margin: 0.986


✅ **CHECKPOINT 8 COMPLETE** - Corpus labeled with BERTJE

**Resume**: Load from `Bertje_labeling/bert_labeled_all.csv`

---
# CHECKPOINT 9: Visualizations
---

Generate interactive visualizations for analysis.


In [28]:
# ============================================================
# CELL 9.1: SETUP VISUALIZATION LIBRARIES
# ============================================================

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    from sklearn.decomposition import PCA
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (14, 8)
    
    print("✓ Visualization libraries loaded")
    VIZ_AVAILABLE = True
except ImportError as e:
    print(f"⚠ Visualization libraries not available: {e}")
    print("  Install: pip install matplotlib seaborn plotly scikit-learn")
    VIZ_AVAILABLE = False

✓ Visualization libraries loaded


In [33]:
print(f"\n{'='*60}")
print("LOADING SCORED DATA")
print(f"{'='*60}")

# Define where to find the scored data file
scores_path = fs.folders['Bertje_labeling'] / 'bert_labeled_all.csv'

# Check if the file exists and load it
if scores_path.exists():
    # Load the CSV file into a dataframe called df_all_scores
    df_all_scores = pd.read_csv(scores_path)
    print(f"\n✓ Successfully loaded {len(df_all_scores)} scored chunks from:")
    print(f"  {scores_path}")
    
    # Show what columns we have in our data
    print(f"\nColumns available: {list(df_all_scores.columns)}")
else:
    # If file doesn't exist, set df_all_scores to None and warn the user
    df_all_scores = None
    print(f"\n⚠ ERROR: No scored data found at {scores_path}")
    print("  Please run CHECKPOINT 5 (the scoring step) first before running this cell")


LOADING SCORED DATA

✓ Successfully loaded 10164 scored chunks from:
  C:\Users\Home\policy-analysis\workflow_data\Finetuned_Slavery-Slavery-policy_10.28.25_v1\Bertje_labeling\bert_labeled_all.csv

Columns available: ['filename', 'chunk_id', 'sentence_count', 'raw_text', 'text_for_scoring', 'cos_Colonialism', 'cos_Historical_Slavery', 'cos_Modern_Racism_Inequality', 'max_score', 'primary_topic', 'score_margin', 'confidence_level', 'text', 'bert_label', 'bert_confidence', 'bert_margin', 'bert_tier']


In [34]:
# This block adds useful columns to help us work with and display the data
# We create 'short_filename' and 'display_label' to make our data easier to read

# Check if we successfully loaded data in the previous block
if df_all_scores is None:
    print("⚠️  No data available. Cannot prepare helper columns.")
    print("   Please run the 'Load Scored Data' block first and fix any errors.")
else:
    # Store a backup copy of the original dataframe before we make any changes
    df_original = df_all_scores.copy()
    print(f"Original dataset: {len(df_original)} chunks")
    
    # Add 'short_filename' column if it doesn't already exist
    # This extracts just the filename (without the full path) for easier reading
    if 'short_filename' not in df_all_scores.columns:
        df_all_scores['short_filename'] = df_all_scores['filename'].apply(
            lambda x: Path(x).name if pd.notna(x) else 'unknown'
        )
        print("✓ Added 'short_filename' column")
    
    # Add 'display_label' column if it doesn't already exist
    # This creates a readable label combining filename and chunk number
    if 'display_label' not in df_all_scores.columns:
        df_all_scores['display_label'] = df_all_scores.apply(
            lambda row: f"{Path(row['filename']).stem}_{row['chunk_id'].split(':')[-1]}" 
            if pd.notna(row.get('filename')) and pd.notna(row.get('chunk_id'))
            else f"chunk_{row.name}", 
            axis=1
        )
        print("✓ Added 'display_label' column")
    
    print(f"\n✅ Data preparation complete!")

Original dataset: 10164 chunks
✓ Added 'short_filename' column
✓ Added 'display_label' column

✅ Data preparation complete!


In [35]:
# This block sets up all the filter parameters you can adjust
# Change these values to focus on different subsets of your policy texts
# For your thesis: use these filters to focus on specific themes or time periods

print(f"\n{'='*60}")
print("FILTER CONFIGURATION")
print(f"{'='*60}")

# FILTER 1: Confidence Level
# Filter by how confident the model is about its scoring
# Options: 'high', 'low', or None to include all confidence levels
CONFIDENCE_FILTER = None
print(f"  Confidence filter: {CONFIDENCE_FILTER or 'All levels'}")

# FILTER 2: Minimum Score Threshold
# Only keep chunks where the highest topic score is above this value
# Range: 0.0 to 1.0 (0.0 = no filter)
MIN_SCORE_THRESHOLD = 0.6
print(f"  Min score threshold: {MIN_SCORE_THRESHOLD}")

# FILTER 3: Topic-Specific Filters
# Set minimum scores for specific topics (useful for your thesis themes!)
# Example: only show chunks with Colonialism score >= 0.3
TOPIC_FILTERS = {
    # Uncomment and adjust these for your thesis analysis:
    # 'Colonialism': 0.3,              # Policy theme
    # 'Historical slavery': 0.25,       # Slavery theme
    # 'Modern racism& inequality': 0.2, # Reparative policy theme
}
if TOPIC_FILTERS:
    print(f"  Topic-specific filters:")
    for topic, score in TOPIC_FILTERS.items():
        print(f"    - {topic}: >= {score}")

# FILTER 4: Year Range
# Filter documents by year (extracted from filename)
# Useful if your policy texts have years in the filename like "policy_2020.pdf"
YEAR_MIN = None  # Example: 2020
YEAR_MAX = None  # Example: 2023
if YEAR_MIN or YEAR_MAX:
    print(f"  Year filter: {YEAR_MIN or 'any'} - {YEAR_MAX or 'any'}")

# FILTER 5: Text Length (Token Count)
# Filter by approximate length of the text chunk
# Useful to exclude very short or very long chunks
MIN_TOKEN_LENGTH = 0     # Minimum words/tokens
MAX_TOKEN_LENGTH = None  # Maximum words/tokens (None = no limit)
if MIN_TOKEN_LENGTH > 0 or MAX_TOKEN_LENGTH:
    print(f"  Token length: {MIN_TOKEN_LENGTH} - {MAX_TOKEN_LENGTH or 'unlimited'}")

# FILTER 6: Filename Filter
# Only include files whose names contain specific text
# Example: FILENAME_CONTAINS = 'policy' or ['policy', 'report']
FILENAME_CONTAINS = None
if FILENAME_CONTAINS:
    print(f"  Filename contains: {FILENAME_CONTAINS}")

# FILTER 7: Score Margin
# Only keep chunks where the top score is clearly higher than the second-highest
# This helps ensure the primary topic assignment is confident
MIN_SCORE_MARGIN = None  # Example: 0.05
if MIN_SCORE_MARGIN:
    print(f"  Min score margin: {MIN_SCORE_MARGIN}")

print(f"\n{'='*60}")


FILTER CONFIGURATION
  Confidence filter: All levels
  Min score threshold: 0.6



In [ ]:
# ============================================================
# CELL 12: DATA FILTERING FOR VISUALIZATION
# ============================================================
print(f"\n{'='*60}")
print("DATA FILTERING OPTIONS")
print(f"{'='*60}")

 # Load scored data
    scores_path = fs.folders['Cosine_labeling'] / 'scores_all_labeled.csv'
    if scores_path.exists():
        df = pd.read_csv(scores_path)
        print(f"\n✓ Loaded {len(df)} scored chunks")
    else:
        print(f"\n⚠ No scored data found at {scores_path}")
        print("  Please run CHECKPOINT 5 first")
        VIZ_AVAILABLE = False


# Check if we have data to filter
if df_all_scores is None:
    print("⚠️  No data available for filtering. Please run Cell 11 first.")
else:
    # Store original dataframe
    df_original = df_all_scores.copy()
    print(f"Original dataset: {len(df_original)} chunks")
    
    # Add helper columns if they don't exist
    if 'short_filename' not in df_all_scores.columns:
        df_all_scores['short_filename'] = df_all_scores['filename'].apply(
            lambda x: Path(x).name if pd.notna(x) else 'unknown'
        )
    
    if 'display_label' not in df_all_scores.columns:
        df_all_scores['display_label'] = df_all_scores.apply(
            lambda row: f"{Path(row['filename']).stem}_{row['chunk_id'].split(':')[-1]}" 
            if pd.notna(row.get('filename')) and pd.notna(row.get('chunk_id'))
            else f"chunk_{row.name}", 
            axis=1
        )
    
    # ============================================================
    # FILTER PARAMETERS - ADJUST THESE
    # ============================================================
    
    print(f"\n📋 Filter Configuration:")
    
    # Confidence level filtering
    CONFIDENCE_FILTER = None  # Options: 'high', 'low', 'none', or None for all
    print(f"  Confidence filter: {CONFIDENCE_FILTER or 'All levels'}")
    
    # Minimum score threshold (all topics must be above this)
    MIN_SCORE_THRESHOLD = 0.0  # Set to 0.0 to disable
    print(f"  Min score threshold: {MIN_SCORE_THRESHOLD}")
    
    # Filter by specific topic scores (set to None to disable)
    TOPIC_FILTERS = {
        # 'Colonialism': 0.3,           # Minimum score for Colonialism topic
        # 'Historical slavery': 0.25,    # Minimum score for Historical slavery
        # 'Modern racism& inequality': 0.0,
    }
    if TOPIC_FILTERS:
        print(f"  Topic-specific filters: {TOPIC_FILTERS}")
    
    # Year filtering (extract year from filename if present)
    YEAR_MIN = None  # Set to 4-digit year (e.g., 2020) or None to disable
    YEAR_MAX = None  # Set to 4-digit year (e.g., 2023) or None to disable
    if YEAR_MIN or YEAR_MAX:
        print(f"  Year filter: {YEAR_MIN or 'any'} - {YEAR_MAX or 'any'}")
    
    # Token length filtering (approximate tokens = words)
    MIN_TOKEN_LENGTH = 0     # Minimum number of tokens in text_for_scoring
    MAX_TOKEN_LENGTH = None  # Maximum number of tokens (None = no limit)
    if MIN_TOKEN_LENGTH > 0 or MAX_TOKEN_LENGTH:
        print(f"  Token length: {MIN_TOKEN_LENGTH} - {MAX_TOKEN_LENGTH or 'unlimited'}")
    
    # Filter by specific filenames (partial match, case-insensitive)
    FILENAME_CONTAINS = None  # e.g., 'policy' or ['policy', 'report'] or None
    if FILENAME_CONTAINS:
        print(f"  Filename contains: {FILENAME_CONTAINS}")
    
    # Score margin filtering (confidence in primary topic)
    MIN_SCORE_MARGIN = None  # e.g., 0.05 to only show clear distinctions
    if MIN_SCORE_MARGIN:
        print(f"  Min score margin: {MIN_SCORE_MARGIN}")
    
    # ============================================================
    # APPLY FILTERS
    # ============================================================
    print(f"\n{'='*60}")
    print("APPLYING FILTERS")
    print(f"{'='*60}")
    
    df_filtered = df_all_scores.copy()
    
    # 0. Filter by confidence level
    if CONFIDENCE_FILTER is not None and 'confidence_level' in df_filtered.columns:
        initial_count = len(df_filtered)
        df_filtered = df_filtered[df_filtered['confidence_level'] == CONFIDENCE_FILTER]
        print(f"  ✓ Confidence level '{CONFIDENCE_FILTER}': {initial_count} → {len(df_filtered)} chunks")
    
    # 1. Apply minimum score threshold across all topics
    if MIN_SCORE_THRESHOLD > 0:
        initial_count = len(df_filtered)
        if 'max_score' not in df_filtered.columns:
            df_filtered['max_score'] = df_filtered[topic_cols].max(axis=1)
        df_filtered = df_filtered[df_filtered['max_score'] >= MIN_SCORE_THRESHOLD]
        print(f"  ✓ Min score threshold ({MIN_SCORE_THRESHOLD}): {initial_count} → {len(df_filtered)} chunks")
    
    # 2. Apply specific topic score filters
    for topic, min_score in TOPIC_FILTERS.items():
        if min_score > 0:
            initial_count = len(df_filtered)
            col_name = f'cos_{topic}'
            if col_name in df_filtered.columns:
                df_filtered = df_filtered[df_filtered[col_name] >= min_score]
                print(f"  ✓ {topic} >= {min_score}: {initial_count} → {len(df_filtered)} chunks")
            else:
                print(f"  ⚠️  Topic '{topic}' not found in data")
    
    # 3. Filter by score margin
    if MIN_SCORE_MARGIN is not None:
        initial_count = len(df_filtered)
        if 'score_margin' not in df_filtered.columns:
            topic_scores = df_filtered[topic_cols].values
            sorted_scores = np.sort(topic_scores, axis=1)[:, ::-1]
            df_filtered['score_margin'] = sorted_scores[:, 0] - sorted_scores[:, 1]
        df_filtered = df_filtered[df_filtered['score_margin'] >= MIN_SCORE_MARGIN]
        print(f"  ✓ Min score margin ({MIN_SCORE_MARGIN}): {initial_count} → {len(df_filtered)} chunks")
    
    # 4. Extract year from filename and filter
    if YEAR_MIN is not None or YEAR_MAX is not None:
        initial_count = len(df_filtered)
        
        # Extract 4-digit years from filename
        import re
        df_filtered['year'] = df_filtered['filename'].apply(
            lambda x: int(match.group()) if (match := re.search(r'\b(19|20)\d{2}\b', str(x))) else None
        )
        
        if YEAR_MIN is not None:
            df_filtered = df_filtered[
                (df_filtered['year'].isna()) | (df_filtered['year'] >= YEAR_MIN)
            ]
        
        if YEAR_MAX is not None:
            df_filtered = df_filtered[
                (df_filtered['year'].isna()) | (df_filtered['year'] <= YEAR_MAX)
            ]
        
        print(f"  ✓ Year filter ({YEAR_MIN or 'any'}-{YEAR_MAX or 'any'}): {initial_count} → {len(df_filtered)} chunks")
        
        # Show year distribution
        year_dist = df_filtered['year'].value_counts().sort_index()
        if len(year_dist) > 0:
            print(f"    Years found: {dict(year_dist)}")
    
    # 5. Filter by token length
    if MIN_TOKEN_LENGTH > 0 or MAX_TOKEN_LENGTH is not None:
        initial_count = len(df_filtered)
        
        # Approximate token count (split by whitespace)
        if 'token_count' not in df_filtered.columns:
            df_filtered['token_count'] = df_filtered['text_for_scoring'].fillna('').str.split().str.len()
        
        if MIN_TOKEN_LENGTH > 0:
            df_filtered = df_filtered[df_filtered['token_count'] >= MIN_TOKEN_LENGTH]
        
        if MAX_TOKEN_LENGTH is not None:
            df_filtered = df_filtered[df_filtered['token_count'] <= MAX_TOKEN_LENGTH]
        
        print(f"  ✓ Token length ({MIN_TOKEN_LENGTH}-{MAX_TOKEN_LENGTH or 'any'}): {initial_count} → {len(df_filtered)} chunks")
        print(f"    Token range: {df_filtered['token_count'].min():.0f} - {df_filtered['token_count'].max():.0f}")
        print(f"    Mean: {df_filtered['token_count'].mean():.1f}")
    
    # 6. Filter by filename
    if FILENAME_CONTAINS is not None:
        initial_count = len(df_filtered)
        
        # Handle single string or list of strings
        if isinstance(FILENAME_CONTAINS, str):
            search_terms = [FILENAME_CONTAINS]
        else:
            search_terms = FILENAME_CONTAINS
        
        # Create mask for any matching term
        mask = df_filtered['short_filename'].str.lower().str.contains(
            '|'.join(search_terms), case=False, na=False
        )
        df_filtered = df_filtered[mask]
        
        print(f"  ✓ Filename contains '{search_terms}': {initial_count} → {len(df_filtered)} chunks")
    
    # ============================================================
    # FINALIZE AND EXPORT
    # ============================================================
    
    # Create the filtered dataframe for visualization
    df_viz = df_filtered.copy()
    
    print(f"\n{'='*60}")
    print(f"FILTERED DATASET: {len(df_viz)} chunks ({len(df_viz)/len(df_original)*100:.1f}% of original)")
    print(f"{'='*60}")
    
    # Show confidence level distribution if available
    if 'confidence_level' in df_viz.columns:
        print(f"\nConfidence level distribution:")
        for level, count in df_viz['confidence_level'].value_counts().items():
            print(f"  {level}: {count} ({count/len(df_viz)*100:.1f}%)")
    
    # Show topic distribution in filtered data
    print(f"\nTopic score statistics:")
    for topic in topics:
        col = f'cos_{topic}'
        if col in df_viz.columns:
            scores = df_viz[col]
            print(f"  {topic:30s}: mean={scores.mean():.3f}, median={scores.median():.3f}, max={scores.max():.3f}")
    
    # Show primary topic distribution
    if 'primary_topic' in df_viz.columns:
        print(f"\nPrimary topic distribution:")
        for topic, count in df_viz['primary_topic'].value_counts().items():
            print(f"  {topic}: {count} ({count/len(df_viz)*100:.1f}%)")
    
    # Show file distribution
    print(f"\nUnique files: {df_viz['short_filename'].nunique()}")
    if df_viz['short_filename'].nunique() <= 20:
        print(f"\nFiles included:")
        for fname in sorted(df_viz['short_filename'].unique()):
            count = (df_viz['short_filename'] == fname).sum()
            print(f"  • {fname}: {count} chunks")
    
    # Save filtered data
    filtered_data_path = os.path.join(data_folder, "filtered_data_for_viz.csv")
    df_viz.to_csv(filtered_data_path, index=False, encoding='utf-8')
    print(f"\n✓ Filtered data saved to: {filtered_data_path}")
    
    print(f"\n✅ Data ready for visualization!")
    print(f"   Use 'df_viz' dataframe in visualization cells")


DATA FILTERING OPTIONS


NameError: name 'df_all_scores' is not defined

In [29]:
# ============================================================
# CELL 9.2: CLUSTERING CONFIGURATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*60}")
    print("CLUSTERING CONFIGURATION")
    print(f"{'='*60}")
    
    # Clustering parameters
    N_CLUSTERS = 5  # Number of clusters per topic
    MIN_SCORE_THRESHOLD = 0.30  # Minimum score to include in clustering
    
    # Load scored data
    scores_path = fs.folders['Cosine_labeling'] / 'scores_all_labeled.csv'
    if scores_path.exists():
        df = pd.read_csv(scores_path)
        print(f"\n✓ Loaded {len(df)} scored chunks")
    else:
        print(f"\n⚠ No scored data found at {scores_path}")
        print("  Please run CHECKPOINT 5 first")
        VIZ_AVAILABLE = False
    
    if VIZ_AVAILABLE:
        # Get topic columns
        topic_cols = [col for col in df.columns if col.startswith('cos_')]
        print(f"\nTopics found: {len(topic_cols)}")
        for col in topic_cols:
            print(f"  - {col.replace('cos_', '')}")
        
        print(f"\nClustering config:")
        print(f"  Clusters per topic: {N_CLUSTERS}")
        print(f"  Min score threshold: {MIN_SCORE_THRESHOLD}")
else:
    print("⚠ Skipping clustering - libraries not available")


CLUSTERING CONFIGURATION

✓ Loaded 10164 scored chunks

Topics found: 3
  - Colonialism
  - Historical_Slavery
  - Modern_Racism_Inequality

Clustering config:
  Clusters per topic: 5
  Min score threshold: 0.3


In [30]:
# ============================================================
# CELL 9.3: PERFORM CLUSTERING PER TOPIC
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*60}")
    print("PERFORMING CLUSTERING PER TOPIC")
    print(f"{'='*60}")
    
    clustering_results = {}
    
    for topic_col in topic_cols:
        topic_name = topic_col.replace("cos_", "")
        print(f"\nProcessing: {topic_name}")
        
        # Filter chunks above threshold
        topic_df = df[df[topic_col] > MIN_SCORE_THRESHOLD].copy()
        
        if len(topic_df) < N_CLUSTERS:
            print(f"  ⚠ Only {len(topic_df)} chunks above threshold, skipping clustering")
            continue
        
        # Prepare features
        X = topic_df[topic_cols].values
        
        # Scale features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # K-means clustering
        n_clusters = min(N_CLUSTERS, len(topic_df))
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X_scaled)
        
        # PCA for visualization
        pca = PCA(n_components=2, random_state=42)
        X_pca = pca.fit_transform(X_scaled)
        
        # Add results to dataframe
        topic_df['cluster'] = cluster_labels
        topic_df['pca_x'] = X_pca[:, 0]
        topic_df['pca_y'] = X_pca[:, 1]
        topic_df['primary_score'] = topic_df[topic_col]
        
        # Create text snippets for hover
        if 'raw_text' in topic_df.columns:
            topic_df['snippet'] = topic_df['raw_text'].apply(
                lambda x: (str(x)[:200] + '...') if pd.notna(x) and len(str(x)) > 200 else str(x) if pd.notna(x) else ''
            )
        else:
            topic_df['snippet'] = ''
        
        # Store results
        clustering_results[topic_name] = {
            'data': topic_df,
            'kmeans': kmeans,
            'pca': pca,
            'scaler': scaler,
            'n_clusters': n_clusters,
            'explained_variance': pca.explained_variance_ratio_
        }
        
        print(f"  ✓ Clustered {len(topic_df)} chunks into {n_clusters} clusters")
        print(f"  PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")
    
    if not clustering_results:
        print("\n⚠ No topics had enough chunks for clustering")
        VIZ_AVAILABLE = False
    else:
        print(f"\n✓ Clustering complete for {len(clustering_results)} topics")


PERFORMING CLUSTERING PER TOPIC

Processing: Colonialism
  ✓ Clustered 9807 chunks into 5 clusters
  PCA explained variance: 87.24%

Processing: Historical_Slavery
  ✓ Clustered 7471 chunks into 5 clusters
  PCA explained variance: 77.61%

Processing: Modern_Racism_Inequality
  ✓ Clustered 8651 chunks into 5 clusters
  PCA explained variance: 81.37%

✓ Clustering complete for 3 topics


In [31]:
# ============================================================
# CELL 9.4: INTERACTIVE 2D CLUSTERING VISUALIZATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*60}")
    print("CREATING 2D CLUSTERING VISUALIZATIONS")
    print(f"{'='*60}")
    
    n_topics = len(clustering_results)
    n_cols = 2
    n_rows = (n_topics + n_cols - 1) // n_cols
    
    fig = make_subplots(
        rows=n_rows, 
        cols=n_cols,
        subplot_titles=list(clustering_results.keys()),
        vertical_spacing=0.10,
        horizontal_spacing=0.10
    )
    
    color_scale = px.colors.qualitative.Set3
    
    for i, (topic_name, results) in enumerate(clustering_results.items()):
        row = i // n_cols + 1
        col = i % n_cols + 1
        
        topic_df = results['data']
        
        # Create hover texts
        hover_texts = {}
        for idx, r in topic_df.iterrows():
            file_name = Path(r.get('filename', '')).name if 'filename' in r else r.get('file_path', '')
            if isinstance(file_name, str) and len(file_name) > 50:
                file_name = Path(file_name).name
            hover = (
                f"<b>File:</b> {file_name}<br>"
                f"<b>Chunk:</b> {r.get('chunk_id', r.get('chunk_uid', 'N/A'))}<br>"
                f"<b>Cluster:</b> {r['cluster']}<br>"
                f"<b>Score:</b> {r['primary_score']:.4f}<br>"
                f"<br><b>Text:</b><br>{r['snippet'][:150]}"
            )
            hover_texts[idx] = hover
        
        # Plot each cluster
        for cluster_id in sorted(topic_df['cluster'].unique()):
            cluster_data = topic_df[topic_df['cluster'] == cluster_id]
            cluster_hover = [hover_texts[idx] for idx in cluster_data.index]
            
            fig.add_trace(
                go.Scatter(
                    x=cluster_data['pca_x'],
                    y=cluster_data['pca_y'],
                    mode='markers',
                    marker=dict(
                        size=8,
                        color=color_scale[cluster_id % len(color_scale)],
                        opacity=0.7,
                        line=dict(width=0.5, color='white')
                    ),
                    name=f'C{cluster_id}',
                    hovertemplate='%{hovertext}<extra></extra>',
                    hovertext=cluster_hover,
                    showlegend=(i == 0),
                    legendgroup=f'cluster_{cluster_id}'
                ),
                row=row,
                col=col
            )
        
        # Plot cluster centers
        centers_pca = results['pca'].transform(results['scaler'].transform(results['kmeans'].cluster_centers_))
        fig.add_trace(
            go.Scatter(
                x=centers_pca[:, 0],
                y=centers_pca[:, 1],
                mode='markers',
                marker=dict(
                    size=15,
                    color='red',
                    symbol='x',
                    line=dict(width=2, color='darkred')
                ),
                name='Centers',
                showlegend=(i == 0),
                legendgroup='centers',
                hoverinfo='skip'
            ),
            row=row,
            col=col
        )
    
    fig.update_layout(
        title_text=f'Topic Clustering (K-means, k={N_CLUSTERS})<br><sub>PCA 2D projection | Min score threshold: {MIN_SCORE_THRESHOLD}</sub>',
        height=400 * n_rows,
        width=1400,
        template='plotly_white',
        hovermode='closest'
    )
    
    fig.update_xaxes(title_text='PCA Component 1', showgrid=True)
    fig.update_yaxes(title_text='PCA Component 2', showgrid=True)
    
    output_path = fs.folders['Visuals'] / 'topic_clustering_2d.html'
    fig.write_html(str(output_path))
    print(f"\n✓ Saved: Visuals/topic_clustering_2d.html")
    
    try:
        fig.show()
    except:
        pass


CREATING 2D CLUSTERING VISUALIZATIONS

✓ Saved: Visuals/topic_clustering_2d.html


In [ ]:
# ============================================================
# CELL 9.5: 3D CLUSTERING VISUALIZATION
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*60}")
    print("CREATING 3D VISUALIZATIONS")
    print(f"{'='*60}")
    
    for topic_name, results in clustering_results.items():
        topic_df = results['data']
        
        if len(topic_df) < 3:
            print(f"  ⚠ Skipping {topic_name}: not enough data points")
            continue
        
        # PCA to 3D
        pca_3d = PCA(n_components=3, random_state=42)
        X_scaled = results['scaler'].transform(topic_df[topic_cols].values)
        X_pca_3d = pca_3d.fit_transform(X_scaled)
        
        # Create hover texts
        hover_texts = {}
        for idx, r in topic_df.iterrows():
            file_name = Path(r.get('filename', '')).name if 'filename' in r else r.get('file_path', '')
            if isinstance(file_name, str) and len(file_name) > 50:
                file_name = Path(file_name).name
            hover = (
                f"<b>File:</b> {file_name}<br>"
                f"<b>Cluster:</b> {r['cluster']}<br>"
                f"<b>Score:</b> {r['primary_score']:.4f}<br>"
                f"<b>Text:</b> {r.get('snippet', '')[:100]}"
            )
            hover_texts[idx] = hover
        
        # Create 3D plot
        fig_3d = go.Figure()
        
        for cluster_id in sorted(topic_df['cluster'].unique()):
            cluster_mask = topic_df['cluster'] == cluster_id
            cluster_data = topic_df[cluster_mask]
            cluster_hover = [hover_texts[idx] for idx in cluster_data.index]
            
            fig_3d.add_trace(go.Scatter3d(
                x=X_pca_3d[cluster_mask.values, 0],
                y=X_pca_3d[cluster_mask.values, 1],
                z=X_pca_3d[cluster_mask.values, 2],
                mode='markers',
                marker=dict(
                    size=5,
                    color=color_scale[cluster_id % len(color_scale)],
                    opacity=0.8
                ),
                name=f'Cluster {cluster_id}',
                hovertemplate='%{hovertext}<extra></extra>',
                hovertext=cluster_hover
            ))
        
        fig_3d.update_layout(
            title=f'{topic_name} - 3D Clustering View<br><sub>PCA explained variance: {pca_3d.explained_variance_ratio_.sum():.2%}</sub>',
            scene=dict(
                xaxis_title='PCA 1',
                yaxis_title='PCA 2',
                zaxis_title='PCA 3'
            ),
            width=1000,
            height=800
        )
        
        output_path = fs.folders['Visuals'] / f"topic_clustering_3d_{topic_name.replace(' ', '_')}.html"
        fig_3d.write_html(str(output_path))
        print(f"  ✓ {topic_name} → {output_path.name}")
    
    print(f"\n✓ All 3D visualizations saved")

In [ ]:
# ============================================================
# CELL 9.6: TRAINING METRICS VISUALIZATION (if available)
# ============================================================

if VIZ_AVAILABLE:
    metrics_path = fs.folders['Model_finetuning'] / 'training_metrics.json'
    
    if metrics_path.exists():
        print(f"\n{'='*60}")
        print("CREATING TRAINING METRICS VISUALIZATION")
        print(f"{'='*60}")
        
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        
        fig = go.Figure()
        
        metric_names = ['accuracy', 'precision', 'recall', 'f1']
        metric_values = [metrics.get(f'eval_{m}', 0) for m in metric_names]
        
        fig.add_trace(go.Bar(
            x=[m.capitalize() for m in metric_names],
            y=metric_values,
            marker_color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'],
            text=[f'{v:.3f}' for v in metric_values],
            textposition='outside'
        ))
        
        fig.update_layout(
            title=f"Model Performance Metrics<br><sub>Dataset: {metrics.get('dataset_used', 'N/A')} | Train examples: {metrics.get('num_train_examples', 'N/A')}</sub>",
            yaxis_title="Score",
            yaxis_range=[0, 1.1],
            height=500,
            template='plotly_white'
        )
        
        output_path = fs.folders['Visuals'] / 'training_metrics.html'
        fig.write_html(str(output_path))
        print(f"✓ Saved: Visuals/training_metrics.html")
        
        # Also create a confusion-like heatmap if we have per-topic data
        print(f"\nTraining summary:")
        print(f"  Dataset: {metrics.get('dataset_used', 'N/A')}")
        print(f"  Epochs: {metrics.get('num_epochs', 'N/A')}")
        print(f"  Train examples: {metrics.get('num_train_examples', 'N/A')}")
        print(f"  Val examples: {metrics.get('num_eval_examples', 'N/A')}")
        print(f"  F1 Score: {metrics.get('eval_f1', 0):.4f}")
    else:
        print(f"\n  No training metrics found (model not trained yet)")
    
    fs.save_config("checkpoint9_visuals")
    
    print(f"\n{'='*60}")
    print("ALL VISUALIZATIONS COMPLETE")
    print(f"{'='*60}")
    print(f"\nSaved to: {fs.folders['Visuals']}")
    print(f"\nGenerated files:")
    for file in sorted(fs.folders['Visuals'].glob('*.html')):
        print(f"  - {file.name}")

✅ **CHECKPOINT 9 COMPLETE** - Visualizations generated

**All checkpoints complete!** Check `Visuals/` folder for interactive plots.

---
# Workflow Complete! 🎉
---

## Summary

All checkpoints have been executed:

✅ **CHECKPOINT 0**: Setup & Configuration
✅ **CHECKPOINT 1**: Text Processing
✅ **CHECKPOINT 2**: Vocabulary Building
✅ **CHECKPOINT 3**: Dictionary Expansion
✅ **CHECKPOINT 4**: Topic Vectors
✅ **CHECKPOINT 5**: Chunk Scoring
✅ **CHECKPOINT 6**: Training Data Prep
✅ **CHECKPOINT 7**: Model Training
✅ **CHECKPOINT 8**: BERTJE Labeling
✅ **CHECKPOINT 9**: Visualizations

## Output Location

All outputs saved to: `{workflow_root}`

```
{ModelType}-{Topic}_{Date}_{Version}/
├── config/              # Config snapshots at each checkpoint
├── Dictionary/          # Input, expanded, curated dictionaries
│   └── Dictionary_suggestions/
├── Model_finetuning/    # Trained model + metrics
├── Cosine_labeling/     # Confidence-classified scores
├── Bertje_labeling/     # Model predictions
├── Visuals/             # Interactive HTML visualizations
└── Other_data/          # Chunks, vocabulary, topic vectors
```

## Next Steps

1. **Review Results**: Check visualizations in `Visuals/`
2. **Analyze Model**: Review training metrics
3. **Use Model**: Load trained model for predictions
4. **Iterate**: Adjust config and re-run from any checkpoint

## Using the Trained Model

To use this model in a new workflow:

```python
CONFIG['model']['use_pretrained'] = True
CONFIG['paths']['pretrained_model_path'] = 'path/to/Model_finetuning'
CONFIG['workflow']['model_type'] = 'Finetuned_{Source}'
```

See `WORKFLOW_GUIDE_v3.md` for complete documentation!